# Reasoning Agent: Knowledge Base Entailment

**Arushi Gandhi (M24MA2001)**




In [1]:
import os
import json
import time
import random
import itertools
from pathlib import Path
from collections import Counter

import pandas as pd

TEST_PATH = os.environ.get("TEST_PATH", "test.csv")
OUT_PATH = "submission.csv"
RUN_PUBLIC = os.environ.get("RUN_PUBLIC", "1") != "0"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "submission" else Path.cwd()
random.seed(0)

## 1. Parsing and representation



In [2]:
import re
from functools import lru_cache


TOKEN_SPEC = [
    ("forall", r"∀|\bforall\b|\bFORALL\b|\bForall\b"),
    ("exists", r"∃|\bexists\b|\bEXISTS\b|\bExists\b"),
    ("iff", r"↔|⟷|<->|<=>|\biff\b"),
    ("imp", r"→|—>|-->|->|=>|⇒|\bimplies\b"),
    ("assign", r":="),
    ("rimp", r":-|←|<-"),
    ("xor", r"⊕|\bxor\b|\bXOR\b"),
    ("and", r"∧|&&|&|\band\b|\bAND\b|\^"),
    ("or", r"∨|\|\||\||\bor\b|\bOR\b"),
    ("neq", r"≠|!="),
    ("not", r"¬|~|!|\bnot\b|\bNOT\b|\\\+"),
    ("eq", r"="),
    ("lp", r"[(\[{]"),
    ("rp", r"[)\]}]"),
    ("comma", r","),
    ("dot", r"[.:]"),
    ("name", r"[\w'$][\w'$]*(?:(?:-(?!>)|\.(?=\w))[\w'$]+)*"),
]
TOKEN_RE = re.compile("|".join(f"(?P<{k}>{v})" for k, v in TOKEN_SPEC))
FREE_VAR_RE = re.compile(r"^(?:[u-z]\d*|[A-Z]\d*|_\w*|\?\w+)$")


def tokenize(text):
    out = []
    end = 0
    brackets = []
    for m in TOKEN_RE.finditer(text):
        if text[end:m.start()].strip():
            raise ParseError(f"unrecognised input at {end}: {text[end:m.start()]}")
        if m.lastgroup == "lp":
            brackets.append(m.group())
        elif m.lastgroup == "rp":
            if not brackets or "([{".index(brackets.pop()) != ")]}".index(m.group()):
                raise ParseError("mismatched brackets")
        out.append((m.lastgroup, m.group()))
        end = m.end()
    if text[end:].strip() or brackets:
        raise ParseError("incomplete or unrecognised input")
    return out


class ParseError(Exception):
    pass


class Parser:
    def __init__(self, tokens, prolog=False):
        self.toks = tokens
        self.i = 0
        self.bound = []
        self.prolog = prolog
        self.anonymous = 0

    def peek(self, k=0):
        j = self.i + k
        return self.toks[j][0] if j < len(self.toks) else None

    def take(self, kind=None):
        if self.i >= len(self.toks):
            raise ParseError("unexpected end")
        t = self.toks[self.i]
        if kind and t[0] != kind:
            raise ParseError(f"expected {kind} got {t}")
        self.i += 1
        return t

    def parse(self, lenient=False):
        f = self.iff()
        if self.peek() == "rimp":
            self.take()
            body = self.iff()
            while self.peek() == "comma":
                self.take()
                body = ("and", body, self.iff())
            f = ("imp", body, f)
        while self.peek() == "dot":
            self.take()
        if self.i != len(self.toks) and not (lenient and self.i > len(self.toks) // 2):
            raise ParseError(f"trailing tokens at {self.i}: {self.toks[self.i:self.i+3]}")
        return f

    def iff(self):
        left = self.imp()
        while self.peek() == "iff":
            self.take()
            left = ("iff", left, self.imp())
        return left

    def imp(self):
        left = self.xor()
        if self.peek() == "imp":
            self.take()
            return ("imp", left, self.imp())
        return left

    def xor(self):
        left = self.disj()
        while self.peek() == "xor":
            self.take()
            left = ("xor", left, self.disj())
        return left

    def disj(self):
        left = self.conj()
        while self.peek() == "or" or self.bare_v():
            self.take()
            left = ("or", left, self.conj())
        return left

    def bare_v(self):
        return self.peek() == "name" and self.toks[self.i][1] == "v" and self.peek(1) in ("lp", "not", "name", "forall", "exists")

    def conj(self):
        left = self.unary()
        while self.peek() == "and":
            self.take()
            left = ("and", left, self.unary())
        return left

    def unary(self):
        k = self.peek()
        if k == "not":
            self.take()
            return ("not", self.unary())
        if k in ("forall", "exists"):
            self.take()
            names = [self.take("name")[1]]
            while self.peek() == "comma" and self.peek(1) == "name":
                self.take()
                names.append(self.take("name")[1])
            if self.peek() == "dot":
                self.take()
            self.bound.extend(names)
            if self.peek() in ("lp", "forall", "exists"):
                body = self.unary()
            else:
                body = self.iff()
            del self.bound[-len(names):]
            for n in reversed(names):
                body = ("all" if k == "forall" else "ex", n, body)
            return body
        if k == "lp":
            self.take()
            f = self.iff()
            self.take("rp")
            return f
        return self.atom()

    def atom(self):
        if self.peek() != "name":
            raise ParseError(f"expected atom at {self.i}")
        t1 = self.term()
        if self.peek() in ("eq", "neq"):
            op = self.take()[0]
            t2 = self.term()
            a = ("atom", "=", (t1, t2))
            return a if op == "eq" else ("not", a)
        if t1[0] == "f":
            return ("atom", t1[1], t1[2])
        return ("atom", t1[1], ())

    def term(self):
        name = self.take("name")[1]
        if self.peek() == "lp":
            self.take()
            args = []
            if self.peek() != "rp":
                args.append(self.term())
                while self.peek() == "comma":
                    self.take()
                    args.append(self.term())
            self.take("rp")
            return ("f", name, tuple(args))
        if self.prolog and name == "_":
            self.anonymous += 1
            return ("v", f"§anonymous{self.anonymous}")
        if name in self.bound:
            return ("v", name)
        if self.prolog and (name[0].isupper() or name[0] == "_"):
            return ("v", name)
        return ("c", name)


def fix_parens(text):
    depth = 0
    out = []
    for ch in text:
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            if depth == 0:
                continue
            depth -= 1
        out.append(ch)
    return "".join(out) + ")" * depth


@lru_cache(maxsize=200000)
def parse_formula(text, repair=False):
    text = text.strip().rstrip(".").strip()
    text = re.sub(r"((?:∀|∃|\bforall\s+|\bexists\s+)\s*\w+(?:\s*,\s*\w+)*)\.(?=\w)", r"\1. ", text)
    prolog = ":-" in text
    err = None
    for t in candidates(text) if repair else (text,):
        try:
            f = Parser(tokenize(t), prolog=prolog).parse(False)
            return close_free(f, prolog, repair)
        except ParseError as e:
            err = e
    raise ParseError(f"{err} :: {text}")


def candidates(text):
    yield text
    yield fix_parens(text)
    if text.count("(") < text.count(")"):
        yield "(" * (text.count(")") - text.count("(")) + text


def norm_name(s):
    return s.replace("_", "").replace("-", "").replace("'", "").lower()


def close_free(f, prolog=False, normalize=False):
    free = []
    clean = norm_name if normalize else lambda s: s

    def fix_term(t, bound):
        if t[0] == "c":
            if t[1] not in bound and FREE_VAR_RE.match(t[1]) and (prolog or not t[1][0].isupper() or len(t[1]) == 1):
                if t[1] not in free:
                    free.append(t[1])
                return ("v", t[1])
            return ("c", clean(t[1]))
        if t[0] == "v":
            if t[1] not in bound and t[1] not in free:
                free.append(t[1])
            return t
        return ("f", clean(t[1]), tuple(fix_term(a, bound) for a in t[2]))

    def walk(g, bound):
        op = g[0]
        if op == "atom":
            return ("atom", clean(g[1]) if g[1] != "=" else "=", tuple(fix_term(a, bound) for a in g[2]))
        if op == "not":
            return ("not", walk(g[1], bound))
        if op in ("all", "ex"):
            return (op, g[1], walk(g[2], bound | {g[1]}))
        return (op, walk(g[1], bound), walk(g[2], bound))

    g = walk(f, frozenset())
    for v in reversed(free):
        g = ("all", v, g)
    return g


def show_term(t):
    if t[0] == "f":
        return f"{t[1]}({', '.join(show_term(a) for a in t[2])})"
    return t[1]


def show(f):
    op = f[0]
    if op == "atom":
        if f[1] == "=":
            return f"{show_term(f[2][0])} = {show_term(f[2][1])}"
        name = f[1][:1].upper() + f[1][1:]
        return name if not f[2] else f"{name}({', '.join(show_term(a) for a in f[2])})"
    if op == "not":
        return "¬" + show(f[1])
    if op in ("all", "ex"):
        return ("∀" if op == "all" else "∃") + f[1] + " " + show(f[2])
    sym = {"and": "∧", "or": "∨", "imp": "→", "iff": "↔", "xor": "⊕"}[op]
    return f"({show(f[1])} {sym} {show(f[2])})"

In [3]:
for text in ["∀x (P(x) → Q(x))", "successful(X) :- smart(X), kind(X).", "exists y. P(y) & y != alice"]:
    print(text, "=>", repr(parse_formula(text)))

∀x (P(x) → Q(x)) => ('all', 'x', ('imp', ('atom', 'P', (('v', 'x'),)), ('atom', 'Q', (('v', 'x'),))))
successful(X) :- smart(X), kind(X). => ('all', 'X', ('imp', ('and', ('atom', 'smart', (('v', 'X'),)), ('atom', 'kind', (('v', 'X'),))), ('atom', 'successful', (('v', 'X'),))))
exists y. P(y) & y != alice => ('ex', 'y', ('and', ('atom', 'P', (('v', 'y'),)), ('not', ('atom', '=', (('v', 'y'), ('c', 'alice'))))))


## 2. Classical inference primitives



### Small learned advisors



In [4]:
import math


ROUTER_TREE = {'feature': 3, 'threshold': 2.2132867132867133, 'left': {'action': 'baseline', 'confidence': 1.0, 'samples': 406}, 'right': {'feature': 8, 'threshold': 0.05115089514066496, 'left': {'action': 'baseline', 'confidence': 1.0, 'samples': 13}, 'right': {'action': 'model_first', 'confidence': 0.6111111111111112, 'samples': 18}}}
ROUTER_BOUNDS = [(3.0, 59.0), (1.0, 18.0), (1.0, 9.0), (1.0, 4.785714285714286), (1.0, 6.0), (0.0, 14.0), (1.0, 3.0), (1.0, 2.0), (0.0, 0.16216216216216217), (0.07692307692307693, 0.8333333333333334), (0.014925373134328358, 1.0), (3.0, 33.0), (0.0, 1.0)]
CLAUSE_WEIGHTS = [2.942977628754035, -6.252248360473493, 0.534294871855487, -0.2819053781343444, -1.0006010472484985, -3.9649450639256547, -0.2825820778910908, -2.8259132533716946, -0.6327903269897397, 0.5760658262764807, -0.2740887928597347, -2.8526386513189292]
GUIDANCE_DEFAULT = 'off'
ACTIONS = ('baseline', 'model_first', 'ground_first', 'guided_first')


def guidance_depth(t):
    if t[0] != 'f':
        return 0
    return 1 + max((guidance_depth(a) for a in t[2]), default=0)


def problem_features(kb, goal):
    clauses = kb + goal
    n = len(clauses)
    lengths = [len(c) for c in clauses]
    goal_preds = {p for c in goal for _, p, _ in c}
    variables, functions, constants, predicates = set(), set(), set(), set()
    equality = negatives = literals = overlap = 0
    max_depth = max_variables = 0
    for c in clauses:
        local_variables = set()
        for sign, pred, args in c:
            predicates.add((pred, len(args)))
            literals += 1
            equality += pred == '='
            negatives += not sign
            overlap += pred in goal_preds
            for arg in args:
                max_depth = max(max_depth, guidance_depth(arg))
                stack = [arg]
                while stack:
                    t = stack.pop()
                    if t[0] == 'v':
                        variables.add(t);local_variables.add(t)
                    elif t[0] == 'c':
                        constants.add(t)
                    else:
                        functions.add((t[1], len(t[2])));stack.extend(t[2])
        max_variables = max(max_variables, len(local_variables))
    return [float(n), float(len(goal)), float(max(lengths, default=0)), literals / max(n,1),
            float(len(functions)), float(len(constants)), float(max_variables), float(max_depth),
            equality / max(literals,1), negatives / max(literals,1), overlap / max(literals,1),
            float(len(predicates)), sum(k == 1 for k in lengths) / max(n,1)]


def choose_route(features, mode='auto'):
    if mode == 'off' or (mode == 'auto' and GUIDANCE_DEFAULT == 'off') or ROUTER_TREE is None:
        return 'baseline'
    if len(features) != len(ROUTER_BOUNDS) or any(not math.isfinite(x) for x in features):
        return 'baseline'
    if any(x < lo or x > hi for x, (lo, hi) in zip(features, ROUTER_BOUNDS)):
        return 'baseline'
    node = ROUTER_TREE
    while 'feature' in node:
        node = node['left'] if features[node['feature']] <= node['threshold'] else node['right']
    action = node.get('action', 'baseline')
    if action not in ACTIONS or node.get('confidence',0) < .6 or node.get('samples',0) < 8:
        return 'baseline'
    return action


def clause_features(clause, goal_predicates, from_support=False, derived=False):
    n = len(clause)
    negatives = equality = overlap = arity = variables = functions = depth = 0
    for sign, pred, args in clause:
        negatives += not sign
        equality += pred == '='
        overlap += pred in goal_predicates
        arity += len(args)
        for t in args:
            variables += t[0] == 'v'
            functions += t[0] == 'f'
            if t[0] == 'f':
                depth = max(depth, guidance_depth(t))
    return [1.0, min(n,16)/16, negatives/max(n,1), equality/max(n,1), overlap/max(n,1),
            min(arity,32)/32, variables/max(arity,1), functions/max(arity,1), min(depth,12)/12,
            float(n == 1), float(from_support), float(derived)]


def clause_bonus(features):
    if len(features) != len(CLAUSE_WEIGHTS):
        return 0.0
    score = sum(a*b for a,b in zip(features, CLAUSE_WEIGHTS))
    if not math.isfinite(score):
        return 0.0
    return max(-2.0, min(2.0, score))

In [5]:
import heapq
import itertools
import time
from functools import lru_cache
from collections import defaultdict



def walk(t, s):
    while t[0] == "v" and t in s:
        t = s[t]
    return t


def occurs(v, t, s):
    if t[0] == "c":
        return False
    if t[0] == "v":
        t = walk(t, s)
    if t == v:
        return True
    if t[0] == "f":
        return any(occurs(v, a, s) for a in t[2])
    return False


def unify(a, b, s):
    if a == b:
        return s
    if a[0] == "c" and b[0] == "c":
        return None
    if a[0] == "v":
        a = walk(a, s)
    if b[0] == "v":
        b = walk(b, s)
    if a == b:
        return s
    if a[0] == "v":
        if occurs(a, b, s):
            return None
        s = dict(s)
        s[a] = b
        return s
    if b[0] == "v":
        return unify(b, a, s)
    if a[0] == "f" and b[0] == "f" and a[1] == b[1] and len(a[2]) == len(b[2]):
        for x, y in zip(a[2], b[2]):
            s = unify(x, y, s)
            if s is None:
                return None
        return s
    return None


def unify_args(xs, ys, s):
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        s = unify(x, y, s)
        if s is None:
            return None
    return s


def subst(t, s):
    if not s or t[0] == "c":
        return t
    if t[0] == "v":
        t = walk(t, s)
    if t[0] == "f":
        return ("f", t[1], tuple(subst(a, s) for a in t[2]))
    return t


def subst_lit(l, s):
    return (l[0], l[1], tuple(subst(a, s) for a in l[2]))


def term_vars(t, acc):
    if t[0] == "v":
        acc.add(t)
    elif t[0] == "f":
        for a in t[2]:
            term_vars(a, acc)
    return acc


def lit_str(l):
    name = l[1] if l[1] == "=" else l[1][:1].upper() + l[1][1:]
    body = name if not l[2] else f"{name}({', '.join(show_term(a) for a in l[2])})"
    return body if l[0] else "¬" + body


def elim(f):
    op = f[0]
    if op == "atom":
        return f
    if op == "not":
        return ("not", elim(f[1]))
    if op in ("all", "ex"):
        return (op, f[1], elim(f[2]))
    a, b = elim(f[1]), elim(f[2])
    if op == "imp":
        return ("or", ("not", a), b)
    if op == "iff":
        return ("and", ("or", ("not", a), b), ("or", a, ("not", b)))
    if op == "xor":
        return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (op, a, b)


def nnf(f, neg=False):
    op = f[0]
    if op == "atom":
        return ("lit", not neg, f[1], f[2])
    if op == "not":
        return nnf(f[1], not neg)
    if op in ("all", "ex"):
        q = op if not neg else ("ex" if op == "all" else "all")
        return (q, f[1], nnf(f[2], neg))
    if op == "and":
        return ("or" if neg else "and", nnf(f[1], neg), nnf(f[2], neg))
    if op == "or":
        return ("and" if neg else "or", nnf(f[1], neg), nnf(f[2], neg))
    raise ValueError(op)


class Skolemizer:
    def __init__(self):
        self.n = itertools.count()
        self.prefix = "sk"

    def run(self, f, env=None, univ=()):
        env = env or {}
        op = f[0]
        if op == "lit":
            return ("lit", f[1], f[2], tuple(self.term(a, env) for a in f[3]))
        if op == "all":
            v = ("v", f"{f[1]}_{next(self.n)}")
            return self.run(f[2], {**env, f[1]: v}, univ + (v,))
        if op == "ex":
            k = next(self.n)
            sk = ("f", f"{self.prefix}{k}", univ) if univ else ("c", f"{self.prefix}{k}")
            return self.run(f[2], {**env, f[1]: sk}, univ)
        return (op, self.run(f[1], env, univ), self.run(f[2], env, univ))

    def term(self, t, env):
        if t[0] == "v":
            return env.get(t[1], ("c", t[1]))
        if t[0] == "f":
            return ("f", t[1], tuple(self.term(a, env) for a in t[2]))
        return t


def to_clauses(f, limit=400):
    op = f[0]
    if op == "lit":
        return [frozenset([(f[1], f[2], f[3])])]
    if op == "and":
        return to_clauses(f[1], limit) + to_clauses(f[2], limit)
    if op == "or":
        left, right = to_clauses(f[1], limit), to_clauses(f[2], limit)
        if len(left) * len(right) > limit:
            raise OverflowError("cnf blowup")
        return [a | b for a in left for b in right]
    raise ValueError(op)


def clausify(formulas, sk=None):
    sk = sk or Skolemizer()
    out = []
    for f in formulas:
        g = sk.run(nnf(elim(f)))
        for c in to_clauses(g):
            if not any((not s, p, a) in c for s, p, a in c):
                out.append(c)
    return out


def rename(clause, k):
    m = {}

    def rt(t):
        if t[0] == "v":
            if t not in m:
                m[t] = ("v", f"§renamed{k}_{len(m)}")
            return m[t]
        if t[0] == "f":
            return ("f", t[1], tuple(rt(a) for a in t[2]))
        return t

    return frozenset((s, p, tuple(rt(a) for a in args)) for s, p, args in clause)


def clause_key(c):
    lits = sorted(c, key=lambda l: (l[0], l[1], tuple("_" if a[0] == "v" else a[1] for a in l[2])))
    m = {}

    def rt(t):
        if t[0] == "v":
            if t not in m:
                m[t] = ("v", f"x{len(m) + 1}")
            return m[t]
        if t[0] == "f":
            return ("f", t[1], tuple(rt(a) for a in t[2]))
        return t

    return tuple((s, p, tuple(rt(a) for a in args)) for s, p, args in lits)


def term_depth(t):
    if t[0] == "f":
        return function_depth(t)
    return 0


@lru_cache(maxsize=50000)
def function_depth(t):
    return 1 + max((term_depth(a) for a in t[2]), default=0)


def weight(c):
    return len(c) * 3 + sum(term_depth(a) for l in c for a in l[2])


def factors(c):
    out = []
    lits = list(c)
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            a, b = lits[i], lits[j]
            if a[0] == b[0] and a[1] == b[1]:
                s = unify_args(a[2], b[2], {})
                if s is not None:
                    out.append(frozenset(subst_lit(l, s) for l in c))
    return out


def subsumes(c, d):
    c, d = list(c), list(d)
    if len(c) > len(d):
        return False

    def go(i, s):
        if i == len(c):
            return True
        l = c[i]
        for m in d:
            if m[0] == l[0] and m[1] == l[1]:
                s2 = match_args(l[2], m[2], s)
                if s2 is not None and go(i + 1, s2):
                    return True
        return False

    return go(0, {})


def match_args(xs, ys, s):
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        s = match(x, y, s)
        if s is None:
            return None
    return s


def match(p, t, s):
    if p[0] == "v":
        if p in s:
            return s if s[p] == t else None
        s = dict(s)
        s[p] = t
        return s
    if p[0] == "f":
        if t[0] != "f" or t[1] != p[1] or len(t[2]) != len(p[2]):
            return None
        return match_args(p[2], t[2], s)
    return s if p == t else None


def relevant(kb, sos):
    reach = {(l[0], l[1]) for c in sos for l in c}
    keep, rest = [], list(kb)
    changed = True
    while changed:
        changed = False
        left = []
        for c in rest:
            if any((not s, p) in reach for s, p, _ in c):
                keep.append(c)
                reach |= {(l[0], l[1]) for l in c}
                changed = True
            else:
                left.append(c)
        rest = left
    return keep


class Resolution:
    def __init__(self, max_steps=1200, max_clauses=12000, time_limit=0.5, max_len=12, max_depth=5,
                 filter_relevance=False, learned=False, collect=False, verify=None):
        self.max_steps = max_steps
        self.max_clauses = max_clauses
        self.time_limit = time_limit
        self.max_len = max_len
        self.max_depth = max_depth
        self.filter_relevance = filter_relevance
        self.learned = learned
        self.collect = collect
        self.verify = (learned or collect) if verify is None else verify

    def refute(self, kb, sos, trace=False):
        start = time.perf_counter()
        counter = itertools.count()
        seen = {}
        parents = {}
        short = defaultdict(list)
        by_lit = defaultdict(list)
        raw_seen = set()
        queue = []
        self.steps = 0
        self.status = "UNKNOWN"
        self.pruned = False
        self.verification_failed = False
        self.training_examples = []
        samples = {}
        goal_predicates = {p for c in sos for _, p, _ in c}
        initial = {clause_key(c) for c in kb + sos} if self.verify else set()
        self.effective_depth = max(self.max_depth, max((term_depth(a) for c in kb + sos for l in c for a in l[2]), default=0))
        self.effective_len = max(self.max_len, max((len(c) for c in kb + sos), default=0))

        def push(c, why, from_sos):
            if c in raw_seen:
                return None
            if len(raw_seen) < self.max_clauses * 4:
                raw_seen.add(c)
            if len(c) > self.effective_len or any(term_depth(a) > self.effective_depth for l in c for a in l[2]):
                self.pruned = True
                return None
            k = clause_key(c)
            if k in seen:
                return None
            seen[k] = c
            parents[k] = why
            priority = weight(c) - (2 if from_sos else 0)
            if self.learned or self.collect:
                features = clause_features(c, goal_predicates, from_sos, why[0] in ('res','factor'))
                if self.collect and len(samples) < 5000:
                    samples[k] = features
                if self.learned:
                    priority -= clause_bonus(features)
            heapq.heappush(queue, (priority, next(counter), c, k))
            if not c:
                return k
            return None

        def finish(root):
            if self.verify and not verify_refutation(parents, seen, root, initial):
                self.status = 'UNKNOWN'
                self.verification_failed = True
                return False, None
            self.status = 'PROVED'
            if self.collect:
                used = set()
                stack = [root]
                while stack:
                    key = stack.pop()
                    if key not in used:
                        used.add(key);stack.extend(parents[key][1:])
                for key in used:
                    if key not in samples:
                        why = parents[key]
                        samples[key] = clause_features(seen[key], goal_predicates, why[0] != 'input', why[0] in ('res','factor'))
                self.training_examples = [(features, int(key in used)) for key, features in samples.items()]
            return True, self._proof(parents, root) if trace else None

        for c in relevant(kb, sos) if self.filter_relevance else kb:
            r = push(c, ("input",), False)
            if r is not None:
                return finish(r)
        for c in sos:
            r = push(c, ("goal",), True)
            if r is not None:
                return finish(r)

        while queue:
            self.steps += 1
            if self.steps > self.max_steps or len(seen) > self.max_clauses:
                self.status = "LIMIT_REACHED"
                return False, None
            if time.perf_counter() - start > self.time_limit:
                self.status = "LIMIT_REACHED"
                return False, None
            _, _, given, gkey = heapq.heappop(queue)
            signature = frozenset((s, p, len(args)) for s, p, args in given)
            if any(len(p) < len(given) and sig <= signature and subsumes(p, given)
                   for anchor in signature for p, sig in short.get(anchor, ())):
                continue
            g = rename(given, self.steps)
            new = []
            for sign, pred, arity in signature:
                by_lit[(sign, pred, arity)].append((given, gkey))
            for l in g:
                for other, okey in by_lit[(not l[0], l[1], len(l[2]))]:
                    for m in other:
                        if m[0] != l[0] and m[1] == l[1]:
                            s = unify_args(l[2], m[2], {})
                            if s is None:
                                continue
                            r = frozenset(subst_lit(x, s) for x in (g - {l}) | (other - {m}))
                            if any((not a, b, c) in r for a, b, c in r):
                                continue
                            new.append((r, ("res", gkey, okey)))
            for f in factors(given):
                new.append((f, ("factor", gkey)))
            if len(given) <= 2:
                short[min(signature)].append((given, signature))
            for r, why in new:
                k = push(r, why, True)
                if k is not None:
                    return finish(k)
        self.status = "LIMIT_REACHED" if self.pruned else "UNKNOWN"
        return False, None

    def _proof(self, parents, k):
        num = {}
        lines = []

        def visit(key):
            if key in num:
                return num[key]
            why = parents.get(key, ("input",))
            refs = [visit(p) for p in why[1:]]
            num[key] = len(num) + 1
            txt = " ∨ ".join(lit_str(l) for l in key) or "□ (empty clause)"
            tag = {"input": "kb", "goal": "negated query"}.get(why[0], f"{why[0]} {', '.join(map(str, refs))}")
            lines.append(f"{num[key]:>3}. {txt}   [{tag}]")
            return num[key]

        visit(k)
        return lines


def clause_variant(a, b):
    if len(a) != len(b):
        return False
    if a == b:
        return True
    left = sorted(a, key=lambda l: (l[0], l[1], len(l[2])))
    right = list(b)

    def term_pair(x, y, forward, reverse):
        if x[0] == 'v' or y[0] == 'v':
            if x[0] != 'v' or y[0] != 'v':
                return False
            if x in forward:
                return forward[x] == y
            if y in reverse:
                return False
            forward[x] = y;reverse[y] = x
            return True
        if x[0] != y[0] or x[1] != y[1]:
            return False
        return x[0] != 'f' or (len(x[2]) == len(y[2]) and all(term_pair(u,v,forward,reverse) for u,v in zip(x[2],y[2])))

    def visit(i, remaining, forward, reverse):
        if i == len(left):
            return True
        sign, pred, args = left[i]
        for j in remaining:
            s, p, other = right[j]
            if sign != s or pred != p or len(args) != len(other):
                continue
            f, r = dict(forward), dict(reverse)
            if all(term_pair(x,y,f,r) for x,y in zip(args,other)) and visit(i+1, remaining-{j}, f, r):
                return True
        return False

    return visit(0, set(range(len(right))), {}, {})


def verify_refutation(parents, clauses, root, initial):
    if root not in clauses or clauses[root]:
        return False
    done = set()
    active = set()
    stack = [(root, False)]
    while stack:
        key, expanded = stack.pop()
        if key in done:
            continue
        if key not in parents or key not in clauses:
            return False
        why = parents[key]
        if not expanded:
            if key in active:
                return False
            active.add(key)
            stack.append((key, True))
            stack.extend((p,False) for p in why[1:] if p not in done)
            continue
        active.remove(key)
        child = clauses[key]
        if why[0] in ('input','goal'):
            valid = len(why) == 1 and key in initial
        elif why[0] == 'factor' and len(why) == 2:
            valid = any(clause_variant(c, child) for c in factors(clauses[why[1]]))
        elif why[0] == 'res' and len(why) == 3:
            a, b = rename(clauses[why[1]], '§verifyL'), rename(clauses[why[2]], '§verifyR')
            valid = False
            for l in a:
                for m in b:
                    if l[0] != m[0] and l[1] == m[1]:
                        substitution = unify_args(l[2], m[2], {})
                        if substitution is not None:
                            derived = frozenset(subst_lit(x, substitution) for x in (a-{l}) | (b-{m}))
                            if clause_variant(derived, child):
                                valid = True;break
                if valid:
                    break
        else:
            valid = False
        if not valid:
            return False
        done.add(key)
    return True


def literal_of(f):
    if f[0] == "atom":
        return (True, f[1], f[2])
    if f[0] == "not" and f[1][0] == "atom":
        return (False, f[1][1], f[1][2])
    return None


def conj_literals(f):
    if f[0] == "and":
        a, b = conj_literals(f[1]), conj_literals(f[2])
        return None if a is None or b is None else a + b
    l = literal_of(f)
    return None if l is None else [l]


def to_horn(f):
    while f[0] == "all":
        f = f[2]
    if f[0] == "and":
        a, b = to_horn(f[1]), to_horn(f[2])
        return None if a is None or b is None else a + b
    l = literal_of(f)
    if l is not None:
        return [((), l)]
    if f[0] == "imp":
        body = conj_literals(f[1])
        head = conj_literals(f[2])
        if body is None or head is None:
            return None
        return [(tuple(body), h) for h in head]
    return None


def has_functions(rules):
    for body, head in rules:
        for l in body + (head,):
            if any(a[0] == "f" for a in l[2]):
                return True
    return False


def relevant_rules(rules, preds):
    need = set(preds)
    out = []
    left = list(rules)
    changed = True
    while changed:
        changed = False
        rest = []
        for r in left:
            if r[1][1] in need:
                out.append(r)
                need |= {l[1] for l in r[0]}
                changed = True
            else:
                rest.append(r)
        left = rest
    return out


def stratified(rules):
    edges = defaultdict(set)
    for body, head in rules:
        if not head[0]:
            continue
        for l in body:
            edges[head[1]].add((l[1], not l[0]))
    level = defaultdict(int)
    n = len(edges) + 1
    for _ in range(n + 1):
        changed = False
        for h, deps in edges.items():
            for p, neg in deps:
                need = level[p] + (1 if neg else 0)
                if level[h] < need:
                    level[h] = need
                    changed = True
        if not changed:
            return True
        if max(level.values()) > n:
            return False
    return False


class ForwardChainer:
    def __init__(self, rules, cwa=True):
        self.cwa = cwa
        self.base = set()
        self.rules = []
        self.consts = set()
        for body, head in rules:
            for l in body + (head,):
                for a in l[2]:
                    if a[0] == "c":
                        self.consts.add(a)
            if not body and not any(a[0] == "v" for a in head[2]):
                if not (cwa and not head[0]):
                    self.base.add(head)
            else:
                self.rules.append((body, head))
        self.consts = sorted(self.consts)
        self.iterations = 0
        self.firings = 0

    def matches(self, body, s, index, neg_ref):
        if not body:
            yield s
            return
        j = min(range(len(body)), key=lambda k: (self.cwa and not body[k][0], len(index[(body[k][0], body[k][1])])))
        lit = body[j]
        rest = body[:j] + body[j + 1:]
        if self.cwa and not lit[0]:
            args = tuple(subst(a, s) for a in lit[2])
            free = sorted({a for a in args if a[0] == "v"})
            for vals in itertools.product(self.consts, repeat=len(free)):
                s2 = dict(s)
                s2.update(zip(free, vals))
                if (True, lit[1], tuple(subst(a, s2) for a in args)) not in neg_ref:
                    yield from self.matches(rest, s2, index, neg_ref)
            return
        for args in list(index[(lit[0], lit[1])]):
            s2 = unify_args(lit[2], args, s)
            if s2 is not None:
                yield from self.matches(rest, s2, index, neg_ref)

    def closure(self, neg_ref):
        facts = set(self.base)
        why = {f: ("given",) for f in facts}
        index = defaultdict(set)
        for f in facts:
            index[(f[0], f[1])].add(f[2])
        changed = True
        while changed:
            self.iterations += 1
            changed = False
            for body, head in self.rules:
                for s in list(self.matches(body, {}, index, neg_ref)):
                    h = subst_lit(head, s)
                    free = sorted({a for a in h[2] if a[0] == "v"})
                    for vals in itertools.product(self.consts, repeat=len(free)):
                        g = subst_lit(h, dict(zip(free, vals)))
                        if g in facts or (self.cwa and not g[0]):
                            continue
                        facts.add(g)
                        index[(g[0], g[1])].add(g[2])
                        why[g] = (body, head, s)
                        self.firings += 1
                        changed = True
        return facts, why

    def run(self):
        if not self.cwa or not any(not l[0] for body, _ in self.rules for l in body):
            self.facts, self.why = self.closure(set())
            self.upper = self.facts
            return self
        lower = set()
        while True:
            upper, _ = self.closure(lower)
            new_lower, why = self.closure(upper)
            if new_lower == lower:
                break
            lower = new_lower
        self.facts, self.why, self.upper = lower, why, upper
        return self

    def holds(self, lit):
        if self.cwa:
            pos = (True, lit[1], lit[2])
            return pos in self.facts if lit[0] else pos not in self.upper
        return lit in self.facts

    def explain(self, lit, depth=0, seen=None):
        seen = seen if seen is not None else set()
        pad = "  " * depth
        if self.cwa and not lit[0]:
            return [f"{pad}{lit_str(lit)}  (closed world: {lit_str((True, lit[1], lit[2]))} cannot be derived)"]
        why = self.why.get(lit)
        if why is None:
            return [f"{pad}{lit_str(lit)}  (not derivable)"]
        if why[0] == "given" or lit in seen:
            return [f"{pad}{lit_str(lit)}  (fact)"]
        seen.add(lit)
        body, head, s = why
        rule = " ∧ ".join(lit_str(b) for b in body) + " → " + lit_str(head)
        lines = [f"{pad}{lit_str(lit)}  by rule [{rule}]"]
        for b in body:
            g = subst_lit(b, s)
            free = [a for a in g[2] if a[0] == "v"]
            if not free:
                lines += self.explain(g, depth + 1, seen)
        return lines


class BackwardChainer:
    def __init__(self, rules, cwa=True, max_depth=60):
        self.cwa = cwa
        self.max_depth = max_depth
        self.facts = defaultdict(set)
        self.rules = defaultdict(list)
        self.consts = set()
        for body, head in rules:
            for l in body + (head,):
                for a in l[2]:
                    if a[0] == "c":
                        self.consts.add(a)
            if not body and not any(a[0] == "v" for a in head[2]):
                if not (cwa and not head[0]):
                    self.facts[(head[0], head[1])].add(head[2])
            else:
                self.rules[(head[0], head[1])].append((body, head))
        self.consts = sorted(self.consts)
        self.known = {}
        self.cuts = 0
        self.depth_cuts = 0
        self.calls = 0
        self.k = itertools.count()

    def prove(self, goal):
        return self.check(goal, frozenset(), 0)

    def check(self, g, stack, depth):
        self.calls += 1
        if self.cwa and not g[0]:
            return not self.check((True, g[1], g[2]), stack, depth)
        if g in self.known:
            return self.known[g]
        if g[2] in self.facts[(g[0], g[1])]:
            self.known[g] = True
            return True
        if g in stack or depth > self.max_depth:
            self.cuts += 1
            self.depth_cuts += depth > self.max_depth
            return False
        before = self.cuts
        for _ in self.expand(g, stack | {g}, depth):
            self.known[g] = True
            return True
        if self.cuts == before:
            self.known[g] = False
        return False

    def expand(self, g, stack, depth):
        for body, head in self.rules[(g[0], g[1])]:
            n = next(self.k)
            ren = {v: ("v", f"{v[1]}@{n}") for l in body + (head,) for a in l[2] for v in term_vars(a, set())}
            s = unify_args(tuple(subst(a, ren) for a in head[2]), g[2], {})
            if s is None:
                continue
            body2 = tuple(subst_lit(l, ren) for l in body)
            yield from self.solve(body2, s, stack, depth + 1)

    def answers(self, g, stack, depth):
        out = set()
        for args in self.facts[(g[0], g[1])]:
            s = unify_args(g[2], args, {})
            if s is not None:
                out.add(tuple(subst(a, s) for a in g[2]))
        key = (g[0], g[1], tuple(a if a[0] != "v" else ("v", "_") for a in g[2]))
        if key in stack or depth > self.max_depth:
            self.cuts += 1
            self.depth_cuts += depth > self.max_depth
        else:
            for s in self.expand(g, stack | {key}, depth):
                out.add(tuple(subst(a, s) for a in g[2]))
        for args in out:
            if all(a[0] != "v" for a in args):
                self.known[(g[0], g[1], args)] = True
            yield unify_args(g[2], args, {})

    def solve(self, goals, s, stack, depth):
        if not goals:
            yield s
            return
        j = 0
        for i, l in enumerate(goals):
            if not (self.cwa and not l[0]):
                j = i
                break
        g = subst_lit(goals[j], s)
        rest = goals[:j] + goals[j + 1:]
        free = sorted({a for a in g[2] if a[0] == "v"})
        if not free:
            if self.check(g, stack, depth):
                yield from self.solve(rest, s, stack, depth)
            return
        if self.cwa and not g[0]:
            for vals in itertools.product(self.consts, repeat=len(free)):
                s2 = dict(s)
                s2.update(zip(free, vals))
                if self.check(subst_lit(g, s2), stack, depth):
                    yield from self.solve(rest, s2, stack, depth)
            return
        for s1 in list(self.answers(g, stack, depth)):
            s2 = dict(s)
            s2.update(s1)
            yield from self.solve(rest, s2, stack, depth)


class CausalModel:
    def __init__(self, equations, observations, obs_mode="hybrid"):
        self.eq = equations
        self.obs = observations
        self.obs_mode = obs_mode
        names = set(equations)
        for f in equations.values():
            names |= prop_atoms(f)
        names |= {v for v, _ in observations}
        self.vars = names
        self.roots = sorted(v for v in names if v not in equations)

    def evaluate(self, roots, do):
        val = dict(roots)
        val.update(do)
        for _ in range(len(self.vars) + 2):
            changed = False
            for v, f in self.eq.items():
                if v in do:
                    continue
                r = prop_eval(f, val)
                if r is not None and val.get(v) != r:
                    val[v] = r
                    changed = True
            if not changed:
                break
        if self.vars <= val.keys() and all(v in do or prop_eval(f, val) == val[v] for v, f in self.eq.items()):
            return val
        unknown = sorted(set(self.eq) - set(do))
        if len(unknown) > 16:
            raise ValueError("Causal cycle exceeds fixed-point budget")
        solutions = []
        for bits in itertools.product([False, True], repeat=len(unknown)):
            candidate = {**roots, **dict(zip(unknown, bits)), **do}
            if all(v in do or prop_eval(f, candidate) == candidate[v] for v, f in self.eq.items()):
                solutions.append(candidate)
                if len(solutions) > 1:
                    break
        if len(solutions) != 1:
            raise ValueError("Causal equations do not determine a unique fixed point")
        return solutions[0]

    def query(self, target, do, given):
        cf_values = []
        fallback = []
        for bits in itertools.product([True, False], repeat=len(self.roots)):
            u = dict(zip(self.roots, bits))
            world = self.evaluate(u, {})
            if not all(world.get(v) == b for v, b in given):
                continue
            if self.obs_mode == "intervene":
                cf = self.evaluate(u, dict(list(do) + list(self.obs)))
                cf_values.append((u, cf.get(target, False)))
            elif all(world.get(v) == b for v, b in self.obs):
                cf = self.evaluate(u, dict(do))
                cf_values.append((u, cf.get(target, False)))
            elif self.obs_mode == "hybrid":
                cf = self.evaluate(u, dict(list(do) + list(self.obs)))
                fallback.append((u, cf.get(target, False)))
        worlds = cf_values or fallback
        if not worlds:
            return False, []
        return all(r for _, r in worlds), worlds


def prop_atoms(f):
    if f[0] == "atom":
        return {f[1]}
    if f[0] == "not":
        return prop_atoms(f[1])
    return prop_atoms(f[1]) | prop_atoms(f[2])


def prop_eval(f, val):
    op = f[0]
    if op == "atom":
        return val.get(f[1])
    if op == "not":
        r = prop_eval(f[1], val)
        return None if r is None else not r
    a, b = prop_eval(f[1], val), prop_eval(f[2], val)
    if op == "and":
        if a is False or b is False:
            return False
        return None if a is None or b is None else True
    if op == "or":
        if a is True or b is True:
            return True
        return None if a is None or b is None else False
    if a is None or b is None:
        return None
    if op == "imp":
        return (not a) or b
    if op == "iff":
        return a == b
    if op == "xor":
        return a != b
    raise ValueError(op)

## 3. Tabled rules, definitional CNF, SAT and equality



In [6]:
import itertools
import time
from collections import defaultdict, deque, Counter



class SearchLimit(Exception):
    pass


class TabledChainer(ForwardChainer):
    def __init__(self, rules, cwa=False, constants=(), time_limit=2.0):
        super().__init__(rules, cwa)
        self.consts = sorted(set(self.consts) | set(constants)) or [("c", "$domain")]
        self.deadline = time.perf_counter() + time_limit

    def closure(self, neg_ref):
        facts = set(self.base)
        why = {f: ("given",) for f in facts}
        index = defaultdict(set)
        agenda = deque(sorted(facts))
        depend = defaultdict(list)
        for f in facts:
            index[(f[0], f[1])].add(f[2])

        def fire(body, head, s):
            h = subst_lit(head, s)
            free = sorted({a for a in h[2] if a[0] == "v"})
            for vals in itertools.product(self.consts, repeat=len(free)):
                if self.firings % 256 == 0 and time.perf_counter() > self.deadline:
                    raise SearchLimit("rule evaluation budget reached")
                g = subst_lit(h, dict(zip(free, vals)))
                if g in facts or (self.cwa and not g[0]):
                    continue
                facts.add(g)
                index[(g[0], g[1])].add(g[2])
                why[g] = (body, head, s)
                agenda.append(g)
                self.firings += 1

        for body, head in self.rules:
            positive = [i for i, l in enumerate(body) if l[0] or not self.cwa]
            if not positive:
                for s in self.matches(body, {}, index, neg_ref):
                    fire(body, head, s)
            for i in positive:
                l = body[i]
                depend[(l[0], l[1])].append((body, head, i))
        while agenda:
            self.iterations += 1
            if self.iterations % 128 == 0 and time.perf_counter() > self.deadline:
                raise SearchLimit("rule evaluation budget reached")
            fact = agenda.popleft()
            for body, head, i in depend[(fact[0], fact[1])]:
                s = unify_args(body[i][2], fact[2], {})
                if s is not None:
                    for answer in self.matches(body[:i] + body[i + 1:], s, index, neg_ref):
                        fire(body, head, answer)
        return facts, why


def compact_nnf(f, neg=False):
    op = f[0]
    if op == "atom":
        return ("lit", not neg, f[1], f[2])
    if op == "not":
        return compact_nnf(f[1], not neg)
    if op in ("all", "ex"):
        q = op if not neg else ("ex" if op == "all" else "all")
        return (q, f[1], compact_nnf(f[2], neg))
    if op == "imp":
        return ("and" if neg else "or", compact_nnf(f[1], not neg), compact_nnf(f[2], neg))
    if op in ("iff", "xor"):
        if quantified(f[1]) or quantified(f[2]):
            return compact_nnf(elim(f), neg)
        return (op if not neg else ("xor" if op == "iff" else "iff"),
                compact_nnf(f[1]), compact_nnf(f[2]))
    return (("or" if op == "and" else "and") if neg else op,
            compact_nnf(f[1], neg), compact_nnf(f[2], neg))


def quantified(f):
    if f[0] in ("all", "ex"):
        return True
    if f[0] == "atom":
        return False
    if f[0] == "not":
        return quantified(f[1])
    return quantified(f[1]) or quantified(f[2])


def negate_lit(l):
    return (not l[0], l[1], l[2])


class DefinitionalCNF:
    def __init__(self):
        self.sk = Skolemizer()
        self.sk.prefix = "§sk"
        self.next_aux = 0

    def convert(self, formulas):
        clauses = []
        cache = {}

        def put(lits):
            c = frozenset(lits)
            if not any(negate_lit(l) in c for l in c):
                clauses.append(c)

        def encode(g):
            if g[0] == "lit":
                return (g[1], g[2], g[3])
            if g in cache:
                return cache[g]
            a, b = encode(g[1]), encode(g[2])
            vs = set()
            for l in (a, b):
                for t in l[2]:
                    term_vars(t, vs)
            z = (True, f"§aux{self.next_aux}", tuple(sorted(vs)))
            self.next_aux += 1
            cache[g] = z
            na, nb, nz = negate_lit(a), negate_lit(b), negate_lit(z)
            if g[0] == "and":
                put([nz, a]); put([nz, b]); put([z, na, nb])
            elif g[0] == "or":
                put([z, na]); put([z, nb]); put([nz, a, b])
            else:
                for av, bv in itertools.product([False, True], repeat=2):
                    zv = (av != bv) if g[0] == "xor" else (av == bv)
                    put([na if av else a, nb if bv else b, z if zv else nz])
            return z

        def assert_true(g):
            if g[0] == "and":
                assert_true(g[1]); assert_true(g[2])
            else:
                try:
                    for c in small_cnf(g):
                        put(c)
                except OverflowError:
                    put([encode(g)])

        for f in formulas:
            assert_true(self.sk.run(compact_nnf(f)))
        return clauses


def small_cnf(g, limit=32):
    if g[0] == "lit":
        return [frozenset([(g[1], g[2], g[3])])]
    if g[0] in ("xor", "iff"):
        a, b = g[1], g[2]
        if g[0] == "xor":
            return small_cnf(("and", ("or", a, b), ("or", negate_tree(a), negate_tree(b))), limit)
        return small_cnf(("and", ("or", negate_tree(a), b), ("or", a, negate_tree(b))), limit)
    if g[0] not in ("and", "or"):
        raise OverflowError("definition required")
    left, right = small_cnf(g[1], limit), small_cnf(g[2], limit)
    if g[0] == "and":
        if len(left) + len(right) > limit:
            raise OverflowError("definition required")
        return left + right
    if len(left) * len(right) > limit:
        raise OverflowError("definition required")
    return [a | b for a in left for b in right]


def negate_tree(g):
    if g[0] == "lit":
        return ("lit", not g[1], g[2], g[3])
    if g[0] in ("xor", "iff"):
        return ("iff" if g[0] == "xor" else "xor", g[1], g[2])
    return ("or" if g[0] == "and" else "and", negate_tree(g[1]), negate_tree(g[2]))


def all_terms(clauses):
    out = set()

    def visit(t):
        out.add(t)
        if t[0] == "f":
            for a in t[2]:
                visit(a)

    for c in clauses:
        for l in c:
            for t in l[2]:
                visit(t)
    return out


def ground_clauses(clauses, max_instances=25000, deadline=None, domain=None):
    terms = all_terms(clauses)
    constants = list(domain) if domain is not None else sorted(t for t in terms if t[0] == "c") or [("c", "$domain")]
    exact = not any(t[0] == "f" for t in terms)
    if not any(t[0] == "v" for t in terms):
        exact = True
    out = []
    count = 0
    for c in clauses:
        vs = set()
        for l in c:
            for t in l[2]:
                term_vars(t, vs)
        vs = sorted(vs)
        count += len(constants) ** len(vs)
        if count > max_instances:
            raise SearchLimit("grounding budget reached")
        for vals in itertools.product(constants, repeat=len(vs)):
            if len(out) % 256 == 0 and deadline and time.perf_counter() > deadline:
                raise SearchLimit("grounding time budget reached")
            s = dict(zip(vs, vals))
            g = frozenset(subst_lit(l, s) for l in c)
            if not any(negate_lit(l) in g for l in g):
                out.append(g)
    return out, exact


def finite_countermodel(clauses, size=2, time_limit=0.08, max_instances=10000):
    deadline = time.perf_counter() + time_limit
    domain = tuple(("c", f"§model{i}") for i in range(size))
    terms = all_terms(clauses)
    if size == 1:
        collapsed = [frozenset((s, p, (domain[0],) * len(args)) for s, p, args in c) for c in clauses]
        status, _, _ = ground_sat(collapsed, time_limit=max(0.001, deadline-time.perf_counter()), max_instances=max_instances)
        return status == "SAT"
    constants = {t for t in terms if t[0] == "c"}
    functions = {(t[1], len(t[2])) for t in terms if t[0] == "f"}
    table_size = sum(size ** arity for _, arity in functions)
    if table_size + len(constants) > max_instances:
        raise SearchLimit("finite model table budget reached")
    grounded, _ = ground_clauses(clauses, max_instances, deadline, domain)
    grounded += [frozenset([(False, "=", (a, b))]) for a, b in itertools.combinations(domain, 2)]
    values = set(constants)
    for name, arity in functions:
        values.update(("f", name, args) for args in itertools.product(domain, repeat=arity))
    for term in sorted(values):
        grounded.append(frozenset((True, "=", (term, value)) for value in domain))
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        raise SearchLimit("finite model time budget reached")
    status, _, _ = ground_sat(grounded, time_limit=remaining, max_instances=max_instances * 2)
    return status == "SAT"


class DPLL:
    def __init__(self, clauses, deadline, max_nodes=100000):
        self.clauses = [tuple(sorted(c, key=lambda l: (abs(l), l))) for c in dict.fromkeys(frozenset(c) for c in clauses)
                        if not any(-l in c for l in c)]
        self.n = max((abs(l) for c in self.clauses for l in c), default=0)
        self.values = [0] * (self.n + 1)
        self.trail = []
        self.watches = defaultdict(list)
        self.positions = []
        self.deadline = deadline
        self.max_nodes = max_nodes
        self.nodes = 0

    def value(self, l):
        return self.values[abs(l)] * (1 if l > 0 else -1)

    def assign(self, l):
        if self.value(l) < 0:
            return False
        if not self.value(l):
            self.values[abs(l)] = 1 if l > 0 else -1
            self.trail.append(l)
        return True

    def propagate(self, start):
        while start < len(self.trail):
            false = -self.trail[start]
            start += 1
            waiting = self.watches[false]
            i = 0
            while i < len(waiting):
                ci = waiting[i]
                c = self.clauses[ci]
                pos = self.positions[ci]
                k = 0 if c[pos[0]] == false else 1
                other = c[pos[1 - k]]
                if self.value(other) > 0:
                    i += 1
                    continue
                replacement = next((j for j, l in enumerate(c) if j not in pos and self.value(l) >= 0), None)
                if replacement is not None:
                    pos[k] = replacement
                    waiting[i] = waiting[-1]
                    waiting.pop()
                    self.watches[c[replacement]].append(ci)
                else:
                    if self.value(other) < 0 or not self.assign(other):
                        return False
                    i += 1
        return True

    def search(self, start):
        self.nodes += 1
        if self.nodes > self.max_nodes or time.perf_counter() > self.deadline:
            raise SearchLimit("SAT budget reached")
        if not self.propagate(start):
            return False
        scores = Counter()
        for c in self.clauses:
            if any(self.value(l) > 0 for l in c):
                continue
            pending = [l for l in c if not self.value(l)]
            if not pending:
                return False
            for l in pending:
                scores[l] += 1.0 / len(pending)
        if not scores:
            return True
        chosen = max(scores, key=lambda l: (scores[l], -abs(l), l))
        before = len(self.trail)
        for l in (chosen, -chosen):
            self.assign(l)
            if self.search(before):
                return True
            for t in self.trail[before:]:
                self.values[abs(t)] = 0
            del self.trail[before:]
        return False

    def solve(self):
        for i, c in enumerate(self.clauses):
            if not c:
                return False, {}
            self.positions.append([0, min(1, len(c) - 1)])
            self.watches[c[0]].append(i)
            if len(c) > 1:
                self.watches[c[1]].append(i)
            elif not self.assign(c[0]):
                return False, {}
        sat = self.search(0)
        return sat, {i: self.values[i] > 0 for i in range(1, self.n + 1)} if sat else {}


def equality_conflict(atoms, model):
    terms = all_terms([frozenset((True, p, args) for p, args in atoms)])
    parent = {t: t for t in terms}

    def find(t):
        while parent[t] != t:
            parent[t] = parent[parent[t]]
            t = parent[t]
        return t

    def union(a, b):
        a, b = find(a), find(b)
        if a == b:
            return False
        parent[max(a, b)] = min(a, b)
        return True

    positive = []
    for (p, args), i in atoms.items():
        if p == "=" and model.get(i, False):
            positive.append(i)
            union(*args)
    changed = True
    while changed:
        changed = False
        signatures = {}
        for t in terms:
            if t[0] == "f":
                key = (t[1], tuple(find(a) for a in t[2]))
                if key in signatures:
                    changed |= union(t, signatures[key])
                else:
                    signatures[key] = t
    assumptions = [-i for i in positive]
    pred = {}
    for (p, args), i in atoms.items():
        if p == "=":
            if not model.get(i, False) and find(args[0]) == find(args[1]):
                return assumptions + [i]
        else:
            key = (p, tuple(find(a) for a in args))
            val = model.get(i, False)
            if key in pred and pred[key][0] != val:
                oldval, oldi = pred[key]
                return assumptions + ([-oldi, i] if oldval else [oldi, -i])
            pred[key] = (val, i)
    return None


def ground_sat(clauses, time_limit=0.5, max_instances=25000):
    deadline = time.perf_counter() + time_limit
    grounded, exact = ground_clauses(clauses, max_instances, deadline)
    atoms = {}
    encoded = []
    for c in grounded:
        row = []
        for sign, p, args in sorted(c):
            key = (p, args)
            if key not in atoms:
                atoms[key] = len(atoms) + 1
            row.append(atoms[key] if sign else -atoms[key])
        encoded.append(row)
    has_equality = any(p == "=" for p, _ in atoms)
    rounds = 0
    while True:
        sat, model = DPLL(encoded, deadline).solve()
        if not sat:
            return "UNSAT", rounds, None
        if not has_equality:
            return "SAT" if exact else "UNKNOWN", rounds, model
        lemma = equality_conflict(atoms, model)
        if lemma is None:
            return "SAT" if exact else "UNKNOWN", rounds, model
        encoded.append(lemma)
        rounds += 1
        if time.perf_counter() > deadline:
            raise SearchLimit("equality budget reached")


def equality_axioms(clauses):
    terms = all_terms(clauses)
    if not any(p == "=" for c in clauses for _, p, _ in c):
        return []
    x, y, z = [("v", f"$eq{v}") for v in "xyz"]
    eq = lambda a, b: (True, "=", (a, b))
    ne = lambda a, b: (False, "=", (a, b))
    out = [frozenset([eq(x, x)]), frozenset([ne(x, y), eq(y, x)]),
           frozenset([ne(x, y), ne(y, z), eq(x, z)])]
    preds = {(p, len(args)) for c in clauses for _, p, args in c if p != "="}
    funcs = {(t[1], len(t[2])) for t in terms if t[0] == "f"}
    for p, n in sorted(preds | funcs):
        if not n:
            continue
        a = tuple(("v", f"$ea{i}") for i in range(n))
        b = tuple(("v", f"$eb{i}") for i in range(n))
        body = [ne(u, v) for u, v in zip(a, b)]
        if (p, n) in funcs:
            out.append(frozenset(body + [eq(("f", p, a), ("f", p, b))]))
        if (p, n) in preds:
            out.append(frozenset(body + [(False, p, a), (True, p, b)]))
    return out

## 4. Compiling and answering queries



In [7]:
import re
from collections import defaultdict
from functools import lru_cache





CAUSAL_QUERY = re.compile(r"^\s*do\((?P<do>[^)]*)\)\s*(?:\|\s*(?P<given>.*?))?\s*(?:=>|⇒|→|->|□→)\s*(?P<y>.+?)\s*$")


class Result:
    def __init__(self, label, method, trace=None, status=None, diagnostics=None):
        self.label = label
        self.method = method
        self.trace = trace or []
        self.status = status or ("PROVED" if label else "DISPROVED")
        self.diagnostics = diagnostics or []


def signed_atom(text):
    text = text.strip()
    neg = text.startswith(("¬", "~", "!")) or text.lower().startswith("not ")
    name = re.sub(r"^(¬|~|!|not\s+)", "", text, flags=re.I).strip()
    return name, not neg


def solve_causal(premises, query, trace=False, obs_mode="hybrid"):
    m = CAUSAL_QUERY.match(query)
    eqs, obs = {}, []
    for p in premises:
        if ":=" in p:
            v, rhs = p.split(":=", 1)
            f = parse_formula(rhs)
            v = v.strip()
            eqs[v] = f if v not in eqs else ("or", eqs[v], f)
        elif p.strip() and "Bern" not in p:
            obs.append(signed_atom(p))
    eqs = {k: to_prop(f) for k, f in eqs.items()}
    model = CausalModel(eqs, obs, obs_mode)
    do = [signed_atom(x) for x in m["do"].split(",") if x.strip()]
    given = [signed_atom(x) for x in (m["given"] or "").split(",") if x.strip()]
    y, positive = signed_atom(m["y"])
    extra = {v for v, _ in do + given} | {y}
    model.vars |= extra
    model.roots = sorted(model.vars - set(eqs))
    label, worlds = model.query(y, do, given)
    if not positive:
        worlds = [(u, not r) for u, r in worlds]
        label = bool(worlds) and all(r for _, r in worlds)
    lines = []
    if trace:
        lines.append("exogenous: " + ", ".join(model.roots))
        for u, r in worlds[:4]:
            target = y if positive else "not " + y
            lines.append("  u = {" + ", ".join(f"{k}={int(v)}" for k, v in u.items()) + f"}}  ->  {target} = {int(r)}")
    return Result(label, "causal", lines, status=None if worlds else "UNKNOWN")


def to_prop(f):
    if f[0] == "atom":
        return ("atom", f[1], ())
    if f[0] == "not":
        return ("not", to_prop(f[1]))
    if f[0] in ("all", "ex"):
        return to_prop(f[2])
    return (f[0], to_prop(f[1]), to_prop(f[2]))


def parse_all(premises):
    out = []
    for p in premises:
        for line in p.split("\n"):
            if not line.strip():
                continue
            try:
                out.append(parse_formula(line))
            except ParseError:
                pass
    return out


def stem(w):
    for suf in ("ing", "ed", "es", "s"):
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            return w[:-len(suf)]
    return w


def predicate_counts(f, idx, out):
    if f[0] == "atom":
        if f[1] != "=":
            out[(f[1], len(f[2]))].add(idx)
    elif f[0] == "not":
        predicate_counts(f[1], idx, out)
    elif f[0] in ("all", "ex"):
        predicate_counts(f[2], idx, out)
    else:
        predicate_counts(f[1], idx, out)
        predicate_counts(f[2], idx, out)


def merge_variants(formulas):
    seen = defaultdict(set)
    for i, f in enumerate(formulas):
        predicate_counts(f, i, seen)
    groups = defaultdict(list)
    for (name, n), where in seen.items():
        s = stem(name)
        if len(s) >= 4:
            groups[(s, n)].append((len(where), name))
    rename = {}
    for (s, n), names in groups.items():
        if len(names) > 1:
            names.sort(reverse=True)
            for _, name in names[1:]:
                rename[(name, n)] = names[0][1]
    if not rename:
        return formulas
    return [rename_preds(f, rename) for f in formulas]


def rename_preds(f, rename):
    op = f[0]
    if op == "atom":
        return ("atom", rename.get((f[1], len(f[2])), f[1]), f[2])
    if op == "not":
        return ("not", rename_preds(f[1], rename))
    if op in ("all", "ex"):
        return (op, f[1], rename_preds(f[2], rename))
    return (op, rename_preds(f[1], rename), rename_preds(f[2], rename))


def top_conjuncts(f):
    if f[0] == "and":
        return top_conjuncts(f[1]) + top_conjuncts(f[2])
    return [f]


def merge_equal_constants(formulas):
    parent = {}

    def find(c):
        while parent.get(c, c) != c:
            c = parent[c]
        return c

    for f in formulas[:-1]:
        for g in top_conjuncts(f):
            if g[0] == "atom" and g[1] == "=" and all(a[0] == "c" for a in g[2]):
                a, b = find(g[2][0][1]), find(g[2][1][1])
                if a != b:
                    parent[max(a, b)] = min(a, b)
    if not parent:
        return formulas
    return [rename_consts(f, find) for f in formulas]


def rename_consts(f, find):
    op = f[0]
    if op == "atom":
        return ("atom", f[1], tuple(const_term(t, find) for t in f[2]))
    if op == "not":
        return ("not", rename_consts(f[1], find))
    if op in ("all", "ex"):
        return (op, f[1], rename_consts(f[2], find))
    return (op, rename_consts(f[1], find), rename_consts(f[2], find))


def const_term(t, find):
    if t[0] == "c":
        return ("c", find(t[1]))
    if t[0] == "f":
        return ("f", t[1], tuple(const_term(a, find) for a in t[2]))
    return t


def horn_program(formulas):
    rules = []
    for f in formulas:
        r = to_horn(f)
        if r is None:
            return None
        rules += r
    return None if has_functions(rules) else rules


def ground_query(q):
    while q[0] == "all":
        q = q[2]
    lits = []
    stack = [q]
    while stack:
        g = stack.pop()
        if g[0] == "and":
            stack += [g[1], g[2]]
            continue
        l = literal_of(g)
        if l is None or any(a[0] != "c" for a in l[2]):
            return None
        lits.append(l)
    return lits


class CompiledKB:
    def __init__(self, premises, mode="owa", profile="formal"):
        aliases = {"auto": "owa", "classical": "owa", "closed": "cwa"}
        self.mode = aliases.get(mode, mode)
        if self.mode not in ("owa", "cwa"):
            raise ValueError("mode must be owa/classical or cwa/closed")
        if profile not in ("formal", "annotations"):
            raise ValueError("profile must be formal or annotations")
        self.premises = tuple(premises)
        self.profile = profile
        self.formulas = []
        self.errors = []
        self.plans = {}
        self.answers = {}
        self.routing_counts = defaultdict(int)
        for p in premises:
            for line in p.split("\n"):
                if not line.strip():
                    continue
                try:
                    self.formulas.append(parse_formula(line, repair=profile == "annotations"))
                except ParseError as e:
                    self.errors.append(str(e))

    def plan(self, kb):
        key = tuple(kb)
        if key not in self.plans:
            rules = horn_program(kb)
            consts = {a for b, h in rules or [] for l in b + (h,) for a in l[2] if a[0] == "c"}
            self.plans[key] = {"rules": rules, "tables": {}, "backward": {}, "constants": consts, "requests": 0}
        return self.plans[key]

    def answer(self, query, engine="auto", trace=False, prover=None, guidance="auto"):
        if CAUSAL_QUERY.match(query):
            try:
                return solve_causal(self.premises, query, trace)
            except (ParseError, SearchLimit, ValueError) as e:
                return Result(False, "causal", status="UNKNOWN", diagnostics=[str(e)])
        if guidance not in ("auto", "off", "learned", "router", "ranker"):
            raise ValueError("guidance must be auto, off, learned, router or ranker")
        key = (query, engine, trace, guidance)
        if prover is None and key in self.answers:
            return self.answers[key]
        result = self._answer(query, engine, trace, prover, guidance)
        if prover is None:
            self.answers[key] = result
        return result

    def _answer(self, query, engine, trace, prover, guidance):
        if self.errors and self.profile == "formal":
            return Result(False, "parser", status="PARSE_ERROR", diagnostics=self.errors)
        try:
            q = parse_formula(query, repair=self.profile == "annotations")
        except ParseError as e:
            return Result(False, "parser", status="PARSE_ERROR", diagnostics=[str(e)])
        kb = self.formulas
        if self.profile == "annotations":
            *kb, q = merge_variants(kb + [q])
        plan = self.plan(kb)
        rules = plan["rules"]
        goal = ground_query(q)
        if self.mode == "cwa" and (rules is None or goal is None):
            return Result(False, "closed-world", status="UNSUPPORTED",
                          diagnostics=["Closed-world mode requires function-free rules and ground literal queries"])
        definite = rules is not None and all(l[0] and l[1] != "=" for b, h in rules for l in b + (h,))
        if rules is not None and goal is not None and (self.mode == "cwa" or (definite and all(g[1] != "=" for g in goal))):
            consts = tuple(sorted(t for t in all_terms([frozenset(goal)]) if t[0] == "c" and t not in plan["constants"]))
            try:
                plan["requests"] += 1
                if engine in ("auto", "backward") and consts not in plan["tables"] and not trace:
                    if consts not in plan["backward"]:
                        bc = BackwardChainer(rules, cwa=self.mode == "cwa")
                        bc.consts = sorted(set(bc.consts) | set(consts)) or [("c", "$domain")]
                        plan["backward"][consts] = bc
                    bc = plan["backward"][consts]
                    ok = all(bc.prove(g) for g in goal)
                    if bc.cuts == 0:
                        return Result(ok, "backward-" + self.mode, diagnostics=self.errors)
                    del plan["backward"][consts]
                if consts not in plan["tables"]:
                    table = TabledChainer(rules, cwa=self.mode == "cwa", constants=consts).run()
                    plan["tables"][consts] = table
                table = plan["tables"][consts]
                ok = all(table.holds(g) for g in goal)
                lines = []
                if trace:
                    for g in goal:
                        lines += table.explain(g)
                status = "PROVED" if ok else "DISPROVED"
                if self.mode == "cwa" and not ok:
                    values = []
                    for g in goal:
                        pos = (True, g[1], g[2])
                        val = True if pos in table.facts else False if pos not in table.upper else None
                        values.append(val if g[0] or val is None else not val)
                    if False not in values and None in values:
                        status = "UNKNOWN"
                return Result(ok, "tabled-" + self.mode, lines, status, self.errors)
            except (SearchLimit, RecursionError) as e:
                if self.mode == "cwa":
                    return Result(False, "tabled-cwa", status="LIMIT_REACHED", diagnostics=[str(e)])
        if "cnf" not in plan:
            converter = DefinitionalCNF()
            try:
                plan["cnf"] = converter.convert(kb)
                plan["converter"] = converter
            except (RecursionError, OverflowError) as e:
                return Result(False, "clausifier", status="LIMIT_REACHED", diagnostics=[str(e)])
        try:
            neg_goal = plan["converter"].convert([("not", q)])
        except (RecursionError, OverflowError) as e:
            return Result(False, "clausifier", status="LIMIT_REACHED", diagnostics=[str(e)])
        clauses = plan["cnf"]
        diagnostics = list(self.errors)
        if prover is None:
            terms = all_terms(clauses + neg_goal)
            incomplete_grounding = any(t[0] == "f" for t in terms) and any(t[0] == "v" for t in terms)
            action = 'baseline'
            enabled = guidance != 'off' and (guidance != 'auto' or GUIDANCE_DEFAULT != 'off')
            active_guidance = GUIDANCE_DEFAULT if guidance == 'auto' else guidance
            if incomplete_grounding and enabled:
                if active_guidance == 'ranker':
                    action = 'guided_first'
                else:
                    try:
                        action = choose_route(problem_features(clauses,neg_goal), guidance)
                    except (ValueError,RecursionError,OverflowError,IndexError):
                        action = 'baseline'
                    if active_guidance == 'router' and action == 'guided_first':
                        action = 'baseline'
                if action not in ('baseline','model_first','ground_first','guided_first'):
                    action = 'baseline'
            self.routing_counts[action] += 1

            def focused(learned=False):
                search = Resolution(max_steps=300 if learned else 600, time_limit=.04 if learned else .12,
                                    filter_relevance=True, learned=learned)
                ok, proof = search.refute(clauses,neg_goal,trace=trace)
                if search.verification_failed:
                    diagnostics.append('A resolution certificate failed verification')
                if ok:
                    return Result(True,'resolution-guided' if learned else 'resolution-focused',proof or [],'PROVED',diagnostics)
                return None

            def models():
                for size, budget in [(1,.02),(2,.08)]:
                    try:
                        if finite_countermodel(clauses+neg_goal,size,budget):
                            lines = [f"KB and negated query have a {size}-element model"] if trace else []
                            return Result(False,'finite-model',lines,'DISPROVED',diagnostics)
                    except (SearchLimit,RecursionError):
                        pass
                return None

            def ground():
                try:
                    status, rounds, model = ground_sat(clauses+neg_goal,
                        time_limit=.02 if incomplete_grounding else .2,
                        max_instances=2000 if incomplete_grounding else 25000)
                    if status in ('UNSAT','SAT'):
                        lines = ['KB and negated query: '+status,
                                 f'Definitional clauses: {len(clauses)+len(neg_goal)}; equality refinements: {rounds}'] if trace else []
                        return Result(status=='UNSAT','sat-equality' if rounds else 'sat',lines,
                                      'PROVED' if status=='UNSAT' else 'DISPROVED',diagnostics)
                except (SearchLimit,RecursionError) as e:
                    diagnostics.append(str(e))
                return None

            schedules = {'baseline':('focused','models','ground'),
                         'model_first':('models','focused','ground'),
                         'ground_first':('ground','focused','models'),
                         'guided_first':('guided','focused','models','ground')}
            schedule = schedules[action] if incomplete_grounding else ('ground',)
            for stage in schedule:
                result = focused(stage=='guided') if stage in ('focused','guided') else models() if stage=='models' else ground()
                if result is not None:
                    if trace and enabled:
                        result.trace.insert(0,'Search schedule: '+action)
                    return result
        prover = prover or Resolution(time_limit=0.3)
        axioms = equality_axioms(clauses + neg_goal)
        try:
            ok, proof = prover.refute(clauses + axioms, neg_goal, trace=trace)
        except RecursionError:
            return Result(False, "resolution", status="LIMIT_REACHED", diagnostics=diagnostics)
        return Result(ok, "resolution", proof or [], prover.status, diagnostics)

    def answer_many(self, queries, **kwargs):
        return [self.answer(q, **kwargs) for q in queries]


@lru_cache(maxsize=2048)
def _compiled(premises, mode, profile):
    return CompiledKB(premises, mode, profile)


def compile_kb(premises, semantics="classical", profile="formal"):
    if isinstance(premises, str):
        premises = [p for p in re.split(r"\n|;", premises) if p.strip()]
    return _compiled(tuple(premises), semantics, profile)


def solve_logic(premises, query, mode="owa", engine="auto", trace=False, prover=None, profile="formal", guidance="auto"):
    return compile_kb(premises, mode, profile).answer(query, engine, trace, prover, guidance)


def solve(premises, query, mode="owa", engine="auto", trace=False, prover=None, profile="formal", guidance="auto"):
    if isinstance(premises, str):
        premises = [p for p in re.split(r"\n|;", premises) if p.strip()]
    if CAUSAL_QUERY.match(query):
        try:
            return solve_causal(premises, query, trace)
        except (ParseError, ValueError, SearchLimit) as e:
            return Result(False, "causal", status="UNKNOWN", diagnostics=[str(e)])
    return solve_logic(premises, query, mode, engine, trace, prover, profile, guidance)

In [8]:
compiled = compile_kb(["Young(bob)", "Kind(bob)", "Young(x) -> Smart(x)", "Smart(x) & Kind(x) -> Successful(x)"])
for query, result in zip(["Successful(bob)", "~Successful(bob)", "exists x. Successful(x)"],
                         compiled.answer_many(["Successful(bob)", "~Successful(bob)", "exists x. Successful(x)"])):
    print(query, result.label, result.method, result.status)
print("Classical:", solve(["P(alice)"], "~Q(alice)").label)
print("Closed world:", solve(["P(alice)"], "~Q(alice)", mode="cwa").label)

Successful(bob) True backward-owa PROVED
~Successful(bob) False backward-owa DISPROVED
exists x. Successful(x) True sat PROVED
Classical: False
Closed world: True


In [9]:
advice_kb = ["P(alice)", "forall x. P(x) -> exists y. Link(x,y)", "forall x,y. Link(x,y) -> Q(y)"]
for advice in ["off", "router", "ranker", "learned"]:
    yes = solve(advice_kb, "exists z. Q(z)", guidance=advice)
    no = solve(advice_kb, "Missing(alice)", guidance=advice)
    assert yes.label and not no.label
    print(advice, yes.method, yes.status, no.method, no.status)
print("Each guided resolution proof is checked before acceptance")

off resolution-focused PROVED finite-model DISPROVED
router resolution-focused PROVED finite-model DISPROVED
ranker resolution-guided PROVED finite-model DISPROVED
learned resolution-focused PROVED finite-model DISPROVED
Each guided resolution proof is checked before acceptance


## 5. Public data adapters



In [10]:
import re


VAR_WORDS = {"someone", "something", "they", "it", "he", "she", "people", "things", "person", "thing", "them"}


def camel(words):
    words = [w for w in re.split(r"[\s_\-]+", words.strip()) if w]
    if not words:
        return ""
    return words[0].lower() + "".join(w[:1].upper() + w[1:].lower() for w in words[1:])


def pred_name(word):
    return word[:1].upper() + word[1:]


def verb_base(w):
    w = w.lower()
    if w.endswith("ies"):
        return w[:-3] + "y"
    if w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w


def entity(text):
    t = text.strip().lower()
    t = re.sub(r"^(the|a|an)\s+", "", t)
    if t in VAR_WORDS:
        return "x"
    return camel(t)


def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]


def split_and(text):
    parts = re.split(r",\s*and\s+|\s+and\s+|,\s*", text)
    return [p.strip() for p in parts if p.strip()]


def adj_list(text):
    text = re.sub(r"\b(also|always|definitely|very|a|an)\b", "", text)
    return [w.strip() for w in re.split(r",|\band\b", text) if w.strip()]


CLAUSE_IS = re.compile(r"^(?P<s>.+?)\s+(?:is|are)\s+(?P<neg>not\s+)?(?P<a>[\w\s,]+)$")
OBJ = r"(?P<o>(?:the|a|an)\s+\w+(?:\s+\w+)?|[A-Z]\w*|someone|something|they|it|them)"
CLAUSE_VERB_NEG = re.compile(r"^(?P<s>.+?)\s+(?:does|do)\s+not\s+(?P<v>\w+)\s+" + OBJ + "$")
CLAUSE_VERB = re.compile(r"^(?P<s>.+?)\s+(?P<v>\w+)\s+" + OBJ + "$")


def lit(neg, pred, *args):
    a = f"{pred_name(pred)}({', '.join(args)})"
    return "¬" + a if neg else a


def parse_clause(text, prev_subject=None):
    text = text.strip().rstrip(".").strip()
    text = re.sub(r"^(?:then|if)\s+", "", text)
    m = CLAUSE_VERB_NEG.match(text)
    if m:
        return [lit(True, verb_base(m["v"]), entity(m["s"]), entity(m["o"]))], entity(m["s"])
    m = CLAUSE_IS.match(text)
    if m:
        s = entity(m["s"])
        out = [lit(bool(m["neg"]), camel(a), s) for a in adj_list(m["a"])]
        return out, s
    m = CLAUSE_VERB.match(text)
    if m and m["v"].lower() not in ("is", "are"):
        return [lit(False, verb_base(m["v"]), entity(m["s"]), entity(m["o"]))], entity(m["s"])
    if prev_subject is not None:
        t = re.sub(r"^(?:is|are)\s+", "", text)
        neg = t.startswith("not ")
        t = t[4:] if neg else t
        mv = re.match(r"^(?:does\s+not\s+)?(\w+)\s+((?:the|a|an)\s+\w+(?:\s+\w+)?|[A-Z]\w*|someone|something)$", t)
        if mv and len(t.split()) > 1 and "does not" in text:
            return [lit(True, verb_base(mv[1]), prev_subject, entity(mv[2]))], prev_subject
        if mv and len(t.split()) > 1 and not t.startswith("not"):
            return [lit(neg, verb_base(mv[1]), prev_subject, entity(mv[2]))], prev_subject
        return [lit(neg, camel(t), prev_subject)], prev_subject
    raise ValueError(f"cannot parse clause: {text}")


def parse_conditions(text):
    out = []
    subj = None
    for part in re.split(r"\s+and\s+", text):
        part = part.strip()
        if not part:
            continue
        sub = [p.strip() for p in part.split(",") if p.strip()]
        for p in sub:
            lits, subj = parse_clause(p, subj)
            out += lits
    return out


def quantify(body, head):
    text = " ∧ ".join(body) + " → " + " ∧ ".join(head)
    if re.search(r"\(x[,)]|, x\)", text):
        return f"∀x ({text})"
    return text


RULE_IF = re.compile(r"^if\s+(?P<c>.+?),?\s+then\s+(?P<h>.+)$", re.I)
RULE_ALL = re.compile(r"^(?:all\s+)?(?P<a>[\w\s,]+?)\s+(?:people|things|persons|ones)\s+are\s+(?P<neg>not\s+)?(?P<h>[\w\s,]+)$", re.I)
RULE_WHO = re.compile(r"^(?:all\s+)?(?:people|things|someone|something|a person|a thing|anyone|anything|everyone|everything)\s+(?:who|that)\s+(?:is|are)\s+(?P<c>.+?)\s+(?:is|are|will be|must be)\s+(?P<neg>not\s+)?(?P<h>[\w\s,]+)$", re.I)
RULE_ADJ_WHO = re.compile(r"^(?P<a>\w+)\s+(?:people|things)\s+(?:who|that)\s+are\s+(?P<c>.+?)\s+are\s+(?:also\s+)?(?P<neg>not\s+)?(?P<h>[\w\s,]+)$", re.I)


def parse_sentence(s):
    s = s.strip().rstrip(".").strip()
    s = re.sub(r"\s+", " ", s)
    m = RULE_IF.match(s)
    if m:
        body = parse_conditions(m["c"])
        head = parse_conditions(m["h"]) if " and " in m["h"] else parse_clause(m["h"])[0]
        return quantify(body, head)
    m = RULE_ADJ_WHO.match(s)
    if m:
        body = [lit(False, camel(m["a"]), "x")] + [lit(False, camel(a), "x") for a in adj_list(m["c"])]
        head = [lit(bool(m["neg"]), camel(a), "x") for a in adj_list(m["h"])]
        return quantify(body, head)
    m = RULE_WHO.match(s)
    if m:
        body = [lit(False, camel(a), "x") for a in adj_list(m["c"])]
        head = [lit(bool(m["neg"]), camel(a), "x") for a in adj_list(m["h"])]
        return quantify(body, head)
    m = RULE_ALL.match(s)
    if m:
        body = [lit(False, camel(a), "x") for a in adj_list(m["a"])]
        head = [lit(bool(m["neg"]), camel(a), "x") for a in adj_list(m["h"])]
        return quantify(body, head)
    lits, _ = parse_clause(s)
    return " ∧ ".join(lits)


def ruletaker_to_logic(context, question):
    premises = []
    failed = []
    for s in split_sentences(context):
        try:
            premises.append(parse_sentence(s))
        except ValueError:
            failed.append(s)
    query = parse_sentence(question)
    return premises, query, failed


def cb_literal(s):
    s = s.strip()
    neg = s.lower().startswith("not ")
    name = s[4:].strip() if neg else s
    return ("¬" if neg else "") + name


def counterbench_to_logic(given, question):
    eqs = {}
    obs = []
    given = given.replace("We know that", "")
    last = None
    for p in re.split(r"\.\s*", given):
        p = p.strip()
        if not p or "Bern" in p or re.fullmatch(r"[\d)]+", p):
            continue
        m = re.match(r"We observed (.*)", p)
        if m:
            for o in re.split(r"\s+and\s+", m.group(1)):
                o = o.strip()
                flip = re.search(r"\s+not\s+occurs?$", o)
                o = re.sub(r"\s+(not\s+)?occurs?$", "", o)
                l = cb_literal(o)
                if flip:
                    l = l[1:] if l.startswith("¬") else "¬" + l
                obs.append(l)
            continue
        for cl in re.split(r",\s*(?:and\s+)?|^and\s+", p):
            cl = re.sub(r"^and\s+", "", cl.strip())
            if not cl:
                continue
            m = re.match(r"(.+?)\s+(?:together\s+)?causes?\s+(.+)$", cl)
            if m:
                lhs, rhs = m.group(1), m.group(2)
                op = " ∨ " if " or " in lhs else " ∧ "
                ins = re.split(r"\s+or\s+|\s+and\s+", lhs)
                last = op.join(cb_literal(i) for i in ins)
                targets = re.split(r"\s+and\s+", rhs)
            else:
                targets = re.split(r"\s+and\s+", cl)
            if last is None:
                continue
            for t in targets:
                t = cb_literal(t)
                expr = last if len(last.split()) == 1 else f"({last})"
                if t.startswith("¬"):
                    t, expr = t[1:], f"¬{expr}"
                eqs[t] = f"{eqs[t]} ∨ {expr}" if t in eqs else expr
    premises = [f"{v} := {e}" for v, e in eqs.items()] + obs
    q = question.strip()
    m = re.match(r"Would (\w+) occur if not (\w+) instead of (\w+)\?", q)
    if m:
        return premises, f"do(¬{m[2]}) | {m[3]} => {m[1]}"
    m = re.match(r"Would (\w+) occur if not (\w+) and not (\w+)\?", q)
    if m:
        return premises, f"do(¬{m[2]}, ¬{m[3]}) => {m[1]}"
    m = re.match(r"Assume not (\w+), and based on this assumption, further suppose not (\w+)\. Would (\w+) occur\?", q)
    if m:
        return premises, f"do(¬{m[1]}, ¬{m[2]}) => {m[3]}"
    m = re.match(r"Would (\w+) occur if (not )?(\w+)(?: instead of (not )?(\w+))?\?", q)
    if m:
        d = ("¬" if m[2] else "") + m[3]
        g = f" | {'¬' if m[4] else ''}{m[5]}" if m[5] else ""
        return premises, f"do({d}){g} => {m[1]}"
    raise ValueError(q)

In [11]:
def load_public(name, split):
    local = PROJECT_ROOT / "data" / f"folio_{split}.jsonl"
    if name == "folio" and local.exists():
        return [json.loads(line) for line in local.open()]
    cached = Path.home() / ".cache" / "huggingface" / "datasets" / ("tasksource___" + name)
    files = list(cached.glob(f"**/{name}-{split}.arrow")) if cached.exists() else []
    from datasets import Dataset, load_dataset
    if files:
        return Dataset.from_file(str(files[0]))
    return load_dataset("tasksource/" + name, split=split)


def load_counterbench():
    rows = []
    for name in ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]:
        path = PROJECT_ROOT / "data" / "counterbench" / name
        if not path.exists():
            from huggingface_hub import hf_hub_download
            path = Path(hf_hub_download("CounterBench/CounterBench", name, repo_type="dataset"))
        rows += json.loads(path.read_text())
    return rows


def run_evaluation(items, mode="owa", profile="formal"):
    _compiled.cache_clear()
    parse_formula.cache_clear()
    function_depth.cache_clear()
    predictions, elapsed = [], 0.0
    methods, statuses, groups, correct_groups = Counter(), Counter(), Counter(), Counter()
    for premises, query, gold, group in items:
        start = time.perf_counter()
        result = solve(premises, query, mode=mode, profile=profile)
        elapsed += time.perf_counter() - start
        predictions.append(result.label)
        methods[result.method] += 1
        statuses[result.status] += 1
        groups[group] += 1
        correct_groups[group] += result.label == gold
    correct = sum(p == item[2] for p,item in zip(predictions,items))
    summary = {"n":len(items), "correct":correct, "accuracy":correct/len(items),
               "inference_seconds":elapsed, "methods":dict(methods), "statuses":dict(statuses),
               "groups":{k:{"n":n,"correct":correct_groups[k]} for k,n in groups.items()}}
    return summary, predictions

## 6. Experiments and final evaluation



In [12]:
BASELINE = {'FOLIO train': {'n': 1001, 'correct': 808, 'accuracy': 0.8071928071928072, 'inference_seconds': 4.957536485997025, 'methods': {'resolution': 861, 'backward-cwa': 140}, 'statuses': {'LEGACY': 1001}}, 'FOLIO validation': {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 3.021236709001869, 'methods': {'resolution': 181, 'backward-cwa': 22}, 'statuses': {'LEGACY': 203}}, 'CounterBench': {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.017101640000646512, 'methods': {'causal': 1200}, 'statuses': {'LEGACY': 1200}}, 'RuleTaker test': {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 5.169645606078802, 'methods': {'backward-cwa': 150893, 'forward-cwa': 1018}, 'statuses': {'LEGACY': 151911}}}
public_results = {}
public_predictions = {}
public_ids = []
conversion_seconds = 0.0
if RUN_PUBLIC:
    rt_test = load_public("ruletaker", "test")
    folio_train = load_public("folio", "train")
    folio_val = load_public("folio", "validation")
    counterbench = load_counterbench()
    start = time.perf_counter()
    rt_items = []
    for row in rt_test:
        prem, query, failed = ruletaker_to_logic(row["context"], row["question"])
        if failed:
            raise ValueError(failed)
        rt_items.append((prem,query,row["label"]=="entailment",row["config"]))
    cb_items = [(*counterbench_to_logic(x["given_info"],x["question"]),x["answer"]=="yes",x["type"]) for x in counterbench]
    conversion_seconds = time.perf_counter()-start
    train_items = [(x["premises-FOL"].split("\n"),x["conclusion-FOL"],x["label"]=="True",x["label"]) for x in folio_train]
    val_items = [(x["premises-FOL"].split("\n"),x["conclusion-FOL"],x["label"]=="True",x["label"]) for x in folio_val]
    train_result, _ = run_evaluation(train_items, profile="annotations")
    print("FOLIO train:", train_result)
    for name, items, mode, profile in [("RuleTaker test",rt_items,"cwa","formal"),
                                       ("FOLIO validation",val_items,"owa","annotations"),
                                       ("CounterBench",cb_items,"owa","formal")]:
        summary,predictions = run_evaluation(items,mode,profile)
        public_results[name] = summary
        public_predictions[name] = predictions
        print(name, summary)
    public_ids = ([f"ruletaker-test-{i}" for i in range(len(rt_items))]
                  + [f"folio-val-{x['example_id']}" for x in folio_val]
                  + [f"counterbench-{x['type']}-{x['question_id']}-{i}" for i,x in enumerate(counterbench)])
    total_n = sum(v["n"] for v in public_results.values())
    total_correct = sum(v["correct"] for v in public_results.values())
    total_time = sum(v["inference_seconds"] for v in public_results.values())
    print(f"Overall accuracy: {total_correct}/{total_n} = {total_correct/total_n:.10%}")
    print(f"Mean dataset accuracy: {sum(v['accuracy'] for v in public_results.values())/3:.10%}")
    print(f"Total inference time: {total_time:.9f}s")
    print(f"English conversion time: {conversion_seconds:.9f}s")
    rows = []
    for name,value in public_results.items():
        old = BASELINE.get(name,{})
        rows.append((name,value["n"],old.get("accuracy"),value["accuracy"],old.get("inference_seconds"),value["inference_seconds"]))
    display(pd.DataFrame(rows, columns=["dataset","queries","baseline accuracy","revised accuracy","baseline seconds","revised seconds"]))
    summary = {"public":public_results,"train":train_result,"baseline":BASELINE,
               "conversion_seconds":conversion_seconds}
    result_path = PROJECT_ROOT / "workspace" / "notebook_results.json"
    if result_path.parent.exists():
        result_path.write_text(json.dumps(summary,indent=2))

FOLIO train: {'n': 1001, 'correct': 807, 'accuracy': 0.8061938061938062, 'inference_seconds': 13.96395716299412, 'methods': {'sat': 792, 'sat-equality': 35, 'backward-owa': 70, 'resolution-focused': 23, 'finite-model': 44, 'resolution': 34, 'tabled-owa': 3}, 'statuses': {'PROVED': 272, 'DISPROVED': 695, 'UNKNOWN': 9, 'LIMIT_REACHED': 25}, 'groups': {'True': {'n': 388, 'correct': 233}, 'False': {'n': 289, 'correct': 267}, 'Uncertain': {'n': 324, 'correct': 307}}}


RuleTaker test {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 2.9048835680569027, 'methods': {'backward-cwa': 132162, 'tabled-cwa': 19749}, 'statuses': {'PROVED': 76426, 'DISPROVED': 75485}, 'groups': {'depth-1': {'n': 17788, 'correct': 17788}, 'depth-5': {'n': 19021, 'correct': 19021}, 'depth-3': {'n': 17982, 'correct': 17982}, 'depth-3ext': {'n': 23222, 'correct': 23222}, 'depth-2': {'n': 17741, 'correct': 17741}, 'depth-3ext-NatLang': {'n': 31017, 'correct': 31017}, 'NatLang': {'n': 7793, 'correct': 7793}, 'depth-0': {'n': 17347, 'correct': 17347}}}


FOLIO validation {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 5.439005211026597, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-focused': 8, 'sat-equality': 6}, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}, 'groups': {'Uncertain': {'n': 69, 'correct': 67}, 'True': {'n': 72, 'correct': 51}, 'False': {'n': 62, 'correct': 58}}}
CounterBench {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.022385183046935708, 'methods': {'causal': 1200}, 'statuses': {'DISPROVED': 620, 'PROVED': 580}, 'groups': {'basic': {'n': 250, 'correct': 250}, 'conditional': {'n': 250, 'correct': 230}, 'joint': {'n': 250, 'correct': 240}, 'nested': {'n': 250, 'correct': 240}, 'backdoor': {'n': 200, 'correct': 175}}}
Overall accuracy: 153222/153314 = 99.9399924338%
Mean dataset accuracy: 93.7609469075%
Total inference time: 8.366273962s
English conversion time: 15.231130

,dataset,queries,baseline accuracy,revised accuracy,baseline seconds,revised seconds
0,RuleTaker test,151911,1.000000,1.000000,5.169646,2.904884
1,FOLIO validation,203,0.866995,0.866995,3.021237,5.439005
2,CounterBench,1200,0.945833,0.945833,0.017102,0.022385


### Inference-time optimisation



In [13]:
SPEED_VALIDATION = {'before_median_seconds': 10.765704450942394, 'after_median_seconds': 8.32930168493931, 'reduction_percent': 22.63115040084358, 'public_rounds': 3, 'public_predictions_per_round': 153314, 'changed_labels': 0, 'semantics_unchanged': True, 'original_search_budgets_unchanged': True}
SPEED_DETAILS = {'before': {'version': 'before', 'rounds': [{'RuleTaker test': {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 3.2818483611645206, 'statuses': {'PROVED': 76426, 'DISPROVED': 75485}, 'methods': {'backward-cwa': 17597, 'tabled-cwa': 134314}}, 'FOLIO validation': {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 7.59663878399806, 'statuses': {'LIMIT_REACHED': 19, 'DISPROVED': 119, 'PROVED': 57, 'UNKNOWN': 8}, 'methods': {'resolution': 27, 'sat': 152, 'backward-owa': 6, 'resolution-focused': 8, 'sat-equality': 6, 'tabled-owa': 4}}, 'CounterBench': {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.022138544994959375, 'statuses': {'DISPROVED': 620, 'PROVED': 580}, 'methods': {'causal': 1200}}}, {'RuleTaker test': {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 3.314210757936962, 'statuses': {'PROVED': 76426, 'DISPROVED': 75485}, 'methods': {'backward-cwa': 17597, 'tabled-cwa': 134314}}, 'FOLIO validation': {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 7.429806327998449, 'statuses': {'LIMIT_REACHED': 19, 'DISPROVED': 119, 'PROVED': 57, 'UNKNOWN': 8}, 'methods': {'resolution': 27, 'sat': 152, 'backward-owa': 6, 'resolution-focused': 8, 'sat-equality': 6, 'tabled-owa': 4}}, 'CounterBench': {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.021687365006982873, 'statuses': {'DISPROVED': 620, 'PROVED': 580}, 'methods': {'causal': 1200}}}, {'RuleTaker test': {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 3.3170085980573276, 'statuses': {'PROVED': 76426, 'DISPROVED': 75485}, 'methods': {'backward-cwa': 17597, 'tabled-cwa': 134314}}, 'FOLIO validation': {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 7.404736007004431, 'statuses': {'LIMIT_REACHED': 19, 'DISPROVED': 119, 'PROVED': 57, 'UNKNOWN': 8}, 'methods': {'resolution': 27, 'sat': 152, 'backward-owa': 6, 'resolution-focused': 8, 'sat-equality': 6, 'tabled-owa': 4}}, 'CounterBench': {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.021917081997344212, 'statuses': {'DISPROVED': 620, 'PROVED': 580}, 'methods': {'causal': 1200}}}], 'total_seconds': [10.90062569015754, 10.765704450942394, 10.743661687059102], 'median_seconds': 10.765704450942394}, 'after': {'version': 'after', 'rounds': [{'RuleTaker test': {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 2.962444325033175, 'statuses': {'PROVED': 76426, 'DISPROVED': 75485}, 'methods': {'backward-cwa': 132155, 'tabled-cwa': 19756}}, 'FOLIO validation': {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 5.455425578998074, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-focused': 8, 'sat-equality': 6}}, 'CounterBench': {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.02235012202163489, 'statuses': {'DISPROVED': 620, 'PROVED': 580}, 'methods': {'causal': 1200}}}, {'RuleTaker test': {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 2.943719774058991, 'statuses': {'PROVED': 76426, 'DISPROVED': 75485}, 'methods': {'backward-cwa': 132155, 'tabled-cwa': 19756}}, 'FOLIO validation': {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 5.333346580004218, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-focused': 8, 'sat-equality': 6}}, 'CounterBench': {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.022273987022344954, 'statuses': {'DISPROVED': 620, 'PROVED': 580}, 'methods': {'causal': 1200}}}, {'RuleTaker test': {'n': 151911, 'correct': 151911, 'accuracy': 1.0, 'inference_seconds': 2.907963361937618, 'statuses': {'PROVED': 76426, 'DISPROVED': 75485}, 'methods': {'backward-cwa': 132155, 'tabled-cwa': 19756}}, 'FOLIO validation': {'n': 203, 'correct': 176, 'accuracy': 0.8669950738916257, 'inference_seconds': 5.398415076004312, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-focused': 8, 'sat-equality': 6}}, 'CounterBench': {'n': 1200, 'correct': 1135, 'accuracy': 0.9458333333333333, 'inference_seconds': 0.022923246997379465, 'statuses': {'DISPROVED': 620, 'PROVED': 580}, 'methods': {'causal': 1200}}}], 'total_seconds': [8.440220026052884, 8.299340341085554, 8.32930168493931], 'median_seconds': 8.32930168493931}}
from statistics import median
print("Before median seconds:", SPEED_VALIDATION["before_median_seconds"])
print("After median seconds:", SPEED_VALIDATION["after_median_seconds"])
print("Reduction:", SPEED_VALIDATION["reduction_percent"], "%")
print("Changed prediction labels:", SPEED_VALIDATION["changed_labels"])
display(pd.DataFrame([(name,median(run[name]["inference_seconds"] for run in SPEED_DETAILS["before"]["rounds"]),
                     median(run[name]["inference_seconds"] for run in SPEED_DETAILS["after"]["rounds"]))
                     for name in SPEED_DETAILS["before"]["rounds"][0]],
                     columns=["dataset","before median seconds","after median seconds"]))

Before median seconds: 10.765704450942394
After median seconds: 8.32930168493931
Reduction: 22.63115040084358 %
Changed prediction labels: 0


,dataset,before median seconds,after median seconds
0,RuleTaker test,3.314211,2.943720
1,FOLIO validation,7.429806,5.398415
2,CounterBench,0.021917,0.022350


### Learned routing and clause ranking



In [14]:
LEARNED_EXPERIMENT = {'ablation_train.json': {'off': {'n': 1001, 'correct': 807, 'seconds': 13.598944801015023, 'methods': {'sat': 792, 'sat-equality': 35, 'backward-owa': 70, 'resolution-focused': 23, 'finite-model': 44, 'resolution': 34, 'tabled-owa': 3}, 'statuses': {'PROVED': 272, 'DISPROVED': 695, 'UNKNOWN': 9, 'LIMIT_REACHED': 25}}, 'router': {'n': 1001, 'correct': 807, 'seconds': 13.039181136992738, 'methods': {'sat': 792, 'sat-equality': 35, 'backward-owa': 70, 'resolution-focused': 23, 'finite-model': 44, 'resolution': 34, 'tabled-owa': 3}, 'statuses': {'PROVED': 272, 'DISPROVED': 695, 'UNKNOWN': 9, 'LIMIT_REACHED': 25}}, 'ranker': {'n': 1001, 'correct': 807, 'seconds': 14.216936071998134, 'methods': {'sat': 792, 'sat-equality': 35, 'backward-owa': 70, 'resolution-guided': 23, 'finite-model': 44, 'resolution': 34, 'tabled-owa': 3}, 'statuses': {'PROVED': 272, 'DISPROVED': 695, 'UNKNOWN': 9, 'LIMIT_REACHED': 25}}, 'learned': {'n': 1001, 'correct': 807, 'seconds': 13.020787396019841, 'methods': {'sat': 792, 'sat-equality': 35, 'backward-owa': 70, 'resolution-focused': 23, 'finite-model': 44, 'resolution': 34, 'tabled-owa': 3}, 'statuses': {'PROVED': 272, 'DISPROVED': 695, 'UNKNOWN': 9, 'LIMIT_REACHED': 25}}}, 'ablation_validation.json': {'off': {'n': 203, 'correct': 176, 'seconds': 5.239495576008267, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-focused': 8, 'sat-equality': 6}, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}}, 'router': {'n': 203, 'correct': 176, 'seconds': 5.35999221500424, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-focused': 8, 'sat-equality': 6}, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}}, 'ranker': {'n': 203, 'correct': 176, 'seconds': 5.51106644799529, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-guided': 8, 'sat-equality': 6}, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}}, 'learned': {'n': 203, 'correct': 176, 'seconds': 5.359624500993959, 'methods': {'finite-model': 16, 'resolution': 11, 'sat': 152, 'backward-owa': 10, 'resolution-focused': 8, 'sat-equality': 6}, 'statuses': {'DISPROVED': 135, 'LIMIT_REACHED': 10, 'PROVED': 57, 'UNKNOWN': 1}}}, 'selection.json': {'default': 'off'}}
print("Deployment default:", LEARNED_EXPERIMENT["selection.json"]["default"])
display(pd.DataFrame([(mode, LEARNED_EXPERIMENT["ablation_train.json"][mode]["correct"],
                      LEARNED_EXPERIMENT["ablation_train.json"][mode]["seconds"],
                      value["correct"],value["seconds"])
                     for mode,value in LEARNED_EXPERIMENT["ablation_validation.json"].items()],
                     columns=["guidance","train correct / 1001","train seconds","validation correct / 203","validation seconds"]))

Deployment default: off


,guidance,train correct / 1001,train seconds,validation correct / 203,validation seconds
0,off,807,13.598945,176,5.239496
1,router,807,13.039181,176,5.359992
2,ranker,807,14.216936,176,5.511066
3,learned,807,13.020787,176,5.359625


## 7. Reasoning traces



In [15]:
trace_examples = [
    (["Young(bob)","Kind(bob)","Young(x) -> Smart(x)","Smart(x) & Kind(x) -> Successful(x)"],"Successful(bob)"),
    (["alice = bob","P(f(alice))"],"P(f(bob))"),
    (["Blaf := ~Ziklo","Vork := Blaf & Trune","Lumbo := Vork","Trune"],"do(~Ziklo) | Ziklo => Lumbo")
]
for premises,query in trace_examples:
    result=solve(premises,query,trace=True)
    print(query, result.label, result.method, result.status)
    print("\n".join(result.trace));print()
result=solve(["P(alice) | Q(alice)","~P(alice)"],"Q(alice)",trace=True,
             prover=Resolution(filter_relevance=False))
print("Resolution refutation:")
print("\n".join(result.trace))

Successful(bob) True tabled-owa PROVED
Successful(bob)  by rule [Smart(x) ∧ Kind(x) → Successful(x)]
  Smart(bob)  by rule [Young(x) → Smart(x)]
    Young(bob)  (fact)
  Kind(bob)  (fact)

P(f(bob)) True sat-equality PROVED
KB and negated query: UNSAT
Definitional clauses: 3; equality refinements: 1

do(~Ziklo) | Ziklo => Lumbo True causal PROVED
exogenous: Trune, Ziklo
  u = {Trune=1, Ziklo=1}  ->  Lumbo = 1

Resolution refutation:
  1. P(alice) ∨ Q(alice)   [kb]
  2. ¬P(alice)   [kb]
  3. Q(alice)   [res 1, 2]
  4. ¬Q(alice)   [negated query]
  5. □ (empty clause)   [res 3, 4]


## 8. Test file and submission.csv



In [16]:
import ast
import json
import os
import re
import time

import pandas as pd



ID_COLS = ["id", "example_id", "question_id", "qid", "idx", "uid"]
KB_COLS = ["premises-FOL", "premises_fol", "fol_premises", "premises_logic", "logic_premises",
           "kb", "knowledge_base", "theory", "facts_rules", "premises", "context", "given_info", "facts"]
Q_COLS = ["conclusion-FOL", "conclusion_fol", "fol_conclusion", "query_logic", "logic_query",
          "query", "statement", "conclusion", "hypothesis", "question", "goal"]


def pick(cols, options):
    low = {c.lower(): c for c in cols}
    return next((low[o.lower()] for o in options if o.lower() in low), None)


def as_lines(value):
    if isinstance(value, str) and value.strip().startswith("["):
        try:
            value = ast.literal_eval(value)
        except (SyntaxError, ValueError):
            pass
    if isinstance(value, (list, tuple)):
        return [str(s).strip() for s in value if str(s).strip()]
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    return [s.strip() for s in re.split(r"\n|;|(?<=\))\.\s+(?=\w|[∀∃¬~])", str(value)) if s.strip()]


def predict_row(kb, query, semantics="classical", profile="formal", guidance="auto"):
    premises = as_lines(kb)
    if not isinstance(query, str) or not query.strip():
        return Result(False, "loader", status="PARSE_ERROR", diagnostics=["Missing query"])
    query = query.strip()
    text = " ".join(premises)
    try:
        if query.lower().startswith(("would ", "assume ")) and re.search(r"\bcauses?\b", text):
            prem, q = counterbench_to_logic(text, query)
            return solve(prem, q)
        if re.search(r"\b(?:is|are|does|then)\b", text) and not re.search(r"[∀∃→]|->|:-|\w\(", text):
            prem, q, failed = ruletaker_to_logic(text, query)
            if failed:
                return Result(False, "english-parser", status="PARSE_ERROR", diagnostics=failed)
            return solve(prem, q, mode="cwa")
        return solve(premises, query, mode=semantics, profile=profile, guidance=guidance)
    except (ValueError, TypeError) as e:
        return Result(False, "loader", status="PARSE_ERROR", diagnostics=[str(e)])


def predict_file(path, semantics=None, profile="formal", guidance="auto"):
    path = str(path)
    semantics = semantics or os.environ.get("REASONING_SEMANTICS", "classical")
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        df = pd.read_json(path, lines=path.lower().endswith(".jsonl"))
    id_col = pick(df.columns, ID_COLS)
    kb_col, q_col = pick(df.columns, KB_COLS), pick(df.columns, Q_COLS)
    if kb_col is None or q_col is None:
        raise ValueError("Test file needs premises and query columns. Found: " + ", ".join(df.columns))
    mode_col = pick(df.columns, ["semantics", "reasoning_mode", "world_assumption"])
    profile_col = pick(df.columns, ["parsing_profile"])
    results = []
    start = time.perf_counter()
    for _, row in df.iterrows():
        mode = str(row[mode_col]).strip().lower() if mode_col and pd.notna(row[mode_col]) else semantics
        parsing = str(row[profile_col]).strip().lower() if profile_col and pd.notna(row[profile_col]) else profile
        results.append(predict_row(row[kb_col], row[q_col], mode, parsing, guidance))
    out = pd.DataFrame({"id": list(df[id_col]) if id_col else list(range(len(df))),
                        "label": [r.label for r in results]})
    print(f"{len(out)} predictions in {time.perf_counter() - start:.6f}s")
    print("Statuses:", pd.Series([r.status for r in results]).value_counts().to_dict())
    failures = [(i, r.status, r.diagnostics) for i, r in enumerate(results)
                if r.status not in ("PROVED", "DISPROVED") or r.diagnostics]
    if failures:
        print("Diagnostics (first five):", failures[:5])
    if "label" in df.columns and len(out):
        gold = df["label"].astype(str).str.lower().isin(["true", "entailment", "yes", "1"])
        print(f"Accuracy against provided labels: {(out['label'].to_numpy() == gold.to_numpy()).mean():.10%}")
    return out

In [17]:
import tempfile

demo = pd.DataFrame({"id":[1,2,3,4,5,6],
    "premises":["P(alice); P(x) -> Q(x)","Ready1(alice)","alice = bob; P(f(alice))",
                "P(alice)","P(alice)","['Human(socrates)', 'Human(x) -> Mortal(x)']"],
    "query":["Q(alice)","Ready2(alice)","P(f(bob))","~Q(alice)","~Q(alice)","Mortal(socrates)"],
    "semantics":["classical","classical","classical","classical","cwa","classical"],
    "label":[True,False,True,False,True,True]})
with tempfile.TemporaryDirectory() as tmp:
    demo_path=Path(tmp)/"demo.csv"
    demo.to_csv(demo_path,index=False)
    demo_predictions=predict_file(demo_path)
    assert demo_predictions["label"].tolist()==demo["label"].tolist()
print("Loader checks: 6/6")

6 predictions in 0.000710s
Statuses: {'PROVED': 4, 'DISPROVED': 2}
Accuracy against provided labels: 100.0000000000%
Loader checks: 6/6


In [18]:
if Path(TEST_PATH).exists():
    submission = predict_file(TEST_PATH)
elif RUN_PUBLIC:
    submission = pd.DataFrame({"id":public_ids,"label":[p for name in public_results for p in public_predictions[name]]})
    print("No supplied test file found; writing public evaluation predictions")
else:
    raise FileNotFoundError("Set TEST_PATH to a test file, or enable RUN_PUBLIC")
submission["label"] = submission["label"].map({True:"True",False:"False"})
submission.to_csv(OUT_PATH,index=False)
print(submission.shape, submission["label"].value_counts().to_dict())
submission.head()

No supplied test file found; writing public evaluation predictions
(153314, 2) {'True': 77063, 'False': 76251}


,id,label
0,ruletaker-test-0,True
1,ruletaker-test-1,False
2,ruletaker-test-2,True
3,ruletaker-test-3,False
4,ruletaker-test-4,True
